<a href="https://colab.research.google.com/github/liuminggee/RHESSys/blob/master/plot_extreme_metrics_wateryear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import sys
import duckdb
!pip install kaleido==0.2.1
import geopandas as gpd
import plotly.graph_objects as go
import pyarrow as pa
import gc
import pyarrow.parquet as pq

In [ ]:
def lat_from_id(x):
    lat = (x // 928) * 0.0625 + 25 + 0.0625 / 2.0
    return round(lat,5)

def lon_from_id(x):
    lon = ((x - 207873) % 928) * 0.0625 + (-125.0 + 0.0625 / 2.0)
    return round(lon,5)
def lat_from_id_vec(gridid_series):
    return ((gridid_series // 928) * 0.0625 + 25 + 0.0625 / 2.0).round(5)
def lon_from_id_vec(gridid_series):
    return (((gridid_series - 207873) % 928) * 0.0625 + (-125.0 + 0.0625 / 2.0)).round(5)

def lat_lon_to_id(lat,lon):
    id = ((lat - 25 - 0.0625 / 2) // 0.0625) * 928 + (lon + 125 - 0.0625 / 2.0) / 0.0625 + 1
    return int(id)

def area_vic_grid_from_lat_lon(lat,lon): #1/16th degree
  # Earth's radius in kilometers
  R = 6371
  # Coordinates (in degrees)
  lat1, lat2 = lat - 1./16./2., lat + 1./16./2.
  lon1, lon2 = lon - 1./16./2., lon + 1./16./2.
  # Convert degrees to radians
  phi1, phi2 = np.radians(lat1), np.radians(lat2)
  lambda1, lambda2 = np.radians(lon1), np.radians(lon2)
  # Calculate area
  area = R**2 * abs(lambda2 - lambda1) * abs(np.sin(phi2) - np.sin(phi1))
  return area

def area_vic_grid(gridid): #1/16th degree
  # Earth's radius in kilometers
  R = 6371
  # Coordinates (in degrees)
  lat1, lat2 = lat_from_id(gridid) - 1./16./2., lat_from_id(gridid) + 1./16./2.
  lon1, lon2 = lon_from_id(gridid) - 1./16./2., lon_from_id(gridid) + 1./16./2.
  # Convert degrees to radians
  phi1, phi2 = np.radians(lat1), np.radians(lat2)
  lambda1, lambda2 = np.radians(lon1), np.radians(lon2)
  # Calculate area
  area = R**2 * abs(lambda2 - lambda1) * abs(np.sin(phi2) - np.sin(phi1))
  return area

In [ ]:
# Check available RAM first

import ctypes
# This works on Colab (Linux-based)
ctypes.CDLL("libc.so.6").malloc_trim(0)

import psutil
ram = psutil.virtual_memory()
print(f"Total: {ram.total/1e9:.1f} GB")
print(f"Available: {ram.available/1e9:.1f} GB")

# Loop through each row in df_b
"""
def upsert(df_a, df_b, keys):
    df_a_idx = df_a.set_index(keys)
    df_b_idx = df_b.set_index(keys)
    # Combine with preference to df_b values
    df_combined = df_a_idx.combine_first(df_b_idx)
    df_combined.update(df_b_idx)
    return df_combined.reset_index()
"""
"""
def release_memory(*args):
    for obj in args:
        del obj
    gc.collect()
    # Works on Colab (Linux)
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception as e:
        print(f"malloc_trim skipped: {e}")
"""
def release_memory(*names, scope):
    """Pass variable names as strings + the scope (locals() or globals())"""
    for name in names:
        if name in scope:
            del scope[name]
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception as e:
        print(f"malloc_trim skipped: {e}")

# Monitor memory during the loop
def print_mem():
    ram = psutil.virtual_memory()
    print(f"RAM used: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB ({ram.percent}%)")

def check_mem_and_warn(threshold_pct=85):
    ram = psutil.virtual_memory()
    if ram.percent > threshold_pct:
        print(f"⚠️ WARNING: RAM at {ram.percent}% — consider restarting")

def downcast(df):
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes('object').columns:
        if df[col].nunique() / len(df) < 0.5:   # low cardinality → category
            df[col] = df[col].astype('category')
    return df

In [ ]:
# Weighted mean for all numeric columns
def weighted_mean_vectorized(df, group_cols, numeric_cols, weight_col):
    """Fast vectorized weighted mean - no apply()"""
    df_w = df[numeric_cols].multiply(df[weight_col], axis=0)
    df_w[group_cols + [weight_col]] = df[group_cols + [weight_col]]

    agg = df_w.groupby(group_cols, observed=True)[numeric_cols + [weight_col]].sum()
    for c in numeric_cols:
        agg[c] = agg[c] / agg[weight_col]

    return agg.drop(columns=[weight_col]).reset_index()

In [ ]:
def mapping_df(mdf,ptitle,m_var,b_log10,states,wiras,lat_min,lat_max,lon_min,lon_max,
               color,zmin,zmax,tickvals,ticktext,bshowmap): #mdf has "gridid" AND m_var column for mapping
  sdf = mdf[['gridid',m_var]].copy()

  # Skip if all values are NaN or all zero
  if sdf[m_var].isna().all() or (sdf[m_var] == 0).all():
    print(f'Skipping {ptitle}: all values are NaN or zero')
    return None

  sdf["lat"] = lat_from_id_vec(sdf["gridid"])
  sdf["lon"] = lon_from_id_vec(sdf["gridid"])
  if lat_min is None:
    lat_min = sdf["lat"].min()
  if lat_max is None:
    lat_max = sdf["lat"].max()
  if lon_min is None:
    lon_min = sdf["lon"].min()
  if lon_max is None:
    lon_max = sdf["lon"].max()

  ticklen = len(tickvals)

  df_pivot = sdf.pivot(index='lat', columns='lon', values=m_var)
  if b_log10:
    df_pivot_log = np.log10(df_pivot)
    z_pivot = df_pivot_log
  else:
    z_pivot = df_pivot
  fig = go.Figure()
  # Heatmap
  fig.add_trace(go.Heatmap(
      z=z_pivot.values,
      x=df_pivot.columns,
      y=df_pivot.index,
      colorscale=color,
      zmin=zmin,
      zmax=zmax,
      colorbar=dict(
          title='',#f'{m_var}',
          tickvals=tickvals,
          ticktext=ticktext,
          tickmode='array',
          ticks='outside',
          ticklen=ticklen,
          tickwidth=1.5,
          tickcolor='black',
          tickfont=dict(size=10),
      )
  ))
  # State boundaries
  if states is not None:
    for _, state in states.iterrows():
        geom = state.geometry
        polys = [geom] if geom.geom_type == 'Polygon' else geom.geoms
        for poly in polys:
            x, y = poly.exterior.xy
            fig.add_trace(go.Scatter(
                x=list(x), y=list(y),
                mode='lines',
                line=dict(color='black', width=0.8),
                showlegend=False,
                hoverinfo='skip'
            ))
  # WIRA boundaries
  if wiras is not None:
    for _, wira in wiras.iterrows():
        geom = wira.geometry
        polys = [geom] if geom.geom_type == 'Polygon' else geom.geoms
        for poly in polys:
            x, y = poly.exterior.xy
            fig.add_trace(go.Scatter(
                x=list(x), y=list(y),
                mode='lines',
                line=dict(color='blue', width=0.8),
                showlegend=False,
                hoverinfo='skip'
            ))
  #fig.update_xaxes(range=[lon_min,lon_max])
  #fig.update_yaxes(range=[lat_min,lat_max])
  fig.update_layout(
      title=dict(
        text=ptitle,
        x=0.5,          # center the title
        xanchor='center'
      ),
      xaxis_title='Longitude',
      yaxis_title='Latitude',
      margin=dict(t=40, b=40, l=40, r=40),
      width=600,
      height=400,
      plot_bgcolor='white',
      paper_bgcolor='white',
  )

  fig.update_xaxes(
    range=[lon_min, lon_max],
    showticklabels=True,
    ticks='outside',
    ticklen=5,
    tickwidth=1.5,
    tickcolor='black',
    showline=True,
    linecolor='black',
    mirror=True  # ticks on both sides
  )
  fig.update_yaxes(
      range=[lat_min, lat_max],
      showticklabels=True,
      ticks='outside',
      ticklen=5,
      tickwidth=1.5,
      tickcolor='black',
      showline=True,
      linecolor='black',
      mirror=True  # ticks on both sides
  )
  if bshowmap:
    fig.show()
  return fig


In [ ]:
droot = '/content/drive/MyDrive/CMIP6/ExtremeMetrics_merged'
outroot = '/content/drive/MyDrive/CMIP6/ExtremeMetrics_merged_aggregated'
PNW_gridcell_list = f'{droot}/vic_data_list_PNW.txt'
WA_gridcell_list = f'{droot}/vic_data_list_WA.txt'
vicwira_file = f'{droot}/vic_wira.csv'

os.chdir(droot)
with open(PNW_gridcell_list, 'r') as f:
    PNW = f.read().splitlines() #might use later
with open(WA_gridcell_list, 'r') as f:
    WA = f.read().splitlines()
vicwira = pd.read_csv(vicwira_file)
vicwira_area = vicwira.groupby(['WRIA_NR', 'GRID_CODE'])['Shape_Area'].sum().reset_index()
vicwira_area.rename(columns={'GRID_CODE': 'gridid','Shape_Area':'km2'}, inplace=True)
vicwira_area['km2'] = vicwira_area['km2'] / 10_763_910.4  #ft2 -> km2
vicwira_area['gridid'] = vicwira_area['gridid'].astype('Int64')
vicwira_area['WRIA_NR'] = vicwira_area['WRIA_NR'].astype('Int64')

vicwira_valid_gridids = set(vicwira_area["gridid"].unique())
vicwira_area_indexed = vicwira_area[["gridid", "km2", "WRIA_NR"]].set_index("gridid")

wira_area_km2 = vicwira_area.groupby(["WRIA_NR"])['km2'].sum()

In [ ]:
climates = ["maca_v2"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6","ORNL"]
groups = ["ref","annual","monthly"]

periods = {'ref':[],
           '2000s':[1991,2010],
           '2040s':[2030,2059],
           '2080s':[2070,2099]}

#"ref"
['gridid', 'clim', 'gcm', 'scn', 'version', 'bc', 'p99wet', 'p95wet',
       'RL20', 'RL20_2030_2059', 'RL20_2070_2099', 'lat', 'lon']
#"annual"
['year', 'CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
       'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
       'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
       'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
       'max_cold_spell_length', 'wsdi', 'gridid', 'clim', 'gcm', 'scn',
       'version', 'bc', 'lat', 'lon']

#MACAv3-CMIP6
array(['CNRM-ESM2-1', 'MIROC6', 'CMCC-ESM2', 'IPSL-CM6A-LR', 'CNRM-CM6-1',
       'INM-CM4-8', 'EC-Earth3', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR',
       'EC-Earth3-Veg-LR', 'ACCESS-CM2', 'MIROC-ES2L', 'HadGEM3-GC31-LL',
       'GFDL-ESM4', 'UKESM1-0-LL', 'INM-CM5-0', 'GFDL-CM4', 'CanESM5',
       'MRI-ESM2-0', 'ssp245'], dtype=object)
       

In [ ]:
#DO CESM2 & mpi-esm1-2-hr -MONTHLY ONLY!!! because of huge mem requirements
sgcms = ['mpi-esm1-2-hr','cesm2']
if False:
  for sgcm in sgcms:
    for var in ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]:
      climates = ["WRF_CMIP6"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6"]
      groups = ["monthly"] #["ref","annual","monthly"]

      read_from_aggregated_parquet = False
      #do_cesm2_monthly = True #too big!

      WRIA_final_df = {}  #WRIA mean
      grid_final_df = {}  #grid level mean #1990s: 1981-2010 2030–2059 and 2070–2099 #only for annual and month var group
      WRIA_writer = {}
      grid_writer = {}
      for climate in climates:
        WRIA_final_df[climate] = {}
        grid_final_df[climate] = {}
        WRIA_writer[climate] = {}
        grid_writer[climate] = {}
        for group in groups:
          WRIA_final_df[climate][group] = {}
          grid_final_df[climate][group] = {}
          WRIA_writer[climate][group] = {}
          grid_writer[climate][group] = {}
          for period in periods:
            WRIA_writer[climate][group][period] = None
            grid_writer[climate][group][period] = None

      if read_from_aggregated_parquet == False:

        WRIA_writer = {}
        grid_writer = {}
        for climate in climates:
          WRIA_writer[climate] = {}
          grid_writer[climate] = {}
          for group in groups:
            WRIA_writer[climate][group] = {}
            grid_writer[climate][group] = {}
            for period in periods:
              WRIA_writer[climate][group][period] = None
              grid_writer[climate][group][period] = None

        for climate in climates:
          for group in groups:
            if group == 'ref':
              group_cols = ["WRIA_NR", "gcm", "scn","version","bc"]
              numeric_cols = ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
            elif group == 'annual':
              group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year"]
              numeric_cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                              'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                              'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                              'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                              'max_cold_spell_length', 'wsdi']
            elif group == 'monthly':
              group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year","doy_or_month"]
              numeric_cols = ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]

            #WRIA_results = {}
            #grid_results = {}
            #for period in periods:
            #  WRIA_results[period] = []
            #  grid_results[period] = []


            seperate = False
            for filename in os.listdir(droot): #each file represent various climate data source, gcm model, scn, and group (ref,ann,mon)
              WRIA_results = {}
              grid_results = {}
              for period in periods:
                WRIA_results[period] = []
                grid_results[period] = []

              full_path = os.path.join(droot, filename)
              if climate in filename and f'_{group}_' in filename and '.parquet' in filename and f'_{sgcm}_' in filename:
                print(f'{climate} {group} {full_path}')
                df = downcast(pd.read_parquet(full_path,columns=["gridid","gcm", "scn","version","bc", "year","doy_or_month",var]))
                df = df[df["gcm"] != "ssp245"]  #a bug in previous process. Now remove these extra data.

                #to get WRIA mean
                wdf = df[df["gridid"].isin(vicwira_valid_gridids)]
                wdf = wdf.join(vicwira_area_indexed, on="gridid", how="left")

                WRIA_means = weighted_mean_vectorized(wdf, group_cols, [var], "km2")
                release_memory('wdf',scope=globals())  # free wdf immediately after use
                #WRIA_results.append(WRIA_means)
                if group == 'monthly':
                  for period in [p for p in periods if p != 'ref']:
                    pdf = WRIA_means[WRIA_means['year'].between(periods[period][0], periods[period][1])]
                    pdf_means = pdf.groupby([x for x in group_cols if x not in ['year']], observed=True)[[var]].mean().reset_index()
                    WRIA_results[period].append(pdf_means)
                    release_memory('pdf', 'pdf_means',scope=globals())
                release_memory('WRIA_means',scope=globals())  # free after all periods processed


                #to get grid mean
                if group == 'monthly':
                  for period in [p for p in periods if p != 'ref']:
                    pdf = df[df['year'].between(periods[period][0], periods[period][1])]
                    grid_means = pdf.groupby([x for x in group_cols if x not in ['WRIA_NR','year']] + ['gridid'], observed=True)[[var]].mean().reset_index()
                    grid_results[period].append(grid_means)
                    release_memory('pdf', 'grid_means',scope=globals())

                release_memory('df',scope=globals())

                for period in periods:
                  if len(WRIA_results[period]) > 0:
                    print(f'WRIA: {climate} {group} {period}')
                    tmpdf = pd.concat(WRIA_results[period]).reset_index(drop=True)
                    table = pa.Table.from_pandas(tmpdf)
                    if WRIA_writer[climate][group][period] is None:
                      WRIA_writer[climate][group][period] = pq.ParquetWriter(f'{outroot}/WRIA_{climate}_{group}_{period}_{sgcm}_{var}.parquet', table.schema)
                    WRIA_writer[climate][group][period].write_table(table, row_group_size=50_000)
                    WRIA_results[period] = []
                    release_memory('tmpdf', 'table',scope=globals())

                  if len(grid_results[period]) > 0:
                    print(f'grid: {climate} {group} {period}')
                    tmpdf = pd.concat(grid_results[period]).reset_index(drop=True)
                    table = pa.Table.from_pandas(tmpdf)
                    if grid_writer[climate][group][period] is None:
                      grid_writer[climate][group][period] = pq.ParquetWriter(f'{outroot}/grid_{climate}_{group}_{period}_{sgcm}_{var}.parquet', table.schema)
                    grid_writer[climate][group][period].write_table(table, row_group_size=50_000)
                    grid_results[period] = []
                    release_memory('tmpdf', 'table',scope=globals())
                release_memory('WRIA_results', 'grid_results', scope=globals())
                gc.collect()
                check_mem_and_warn()
            for period in periods:
              if WRIA_writer[climate][group][period]:
                WRIA_writer[climate][group][period].close()
                WRIA_writer[climate][group][period] = None
              if grid_writer[climate][group][period]:
                grid_writer[climate][group][period].close()
                grid_writer[climate][group][period] = None
        gc.collect()


In [ ]:
#ALL WRF GCM EXCEPT cesm2 & mpi-esm1-2-hr!!!

climates = ["WRF_CMIP6"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6"]
groups = ["monthly"] #["ref","annual","monthly"]
read_from_aggregated_parquet = False

if False:
  #do_cesm2_monthly = True #too big!

  WRIA_final_df = {}  #WRIA mean
  grid_final_df = {}  #grid level mean #1990s: 1981-2010 2030–2059 and 2070–2099 #only for annual and month var group
  WRIA_writer = {}
  grid_writer = {}
  for climate in climates:
    WRIA_final_df[climate] = {}
    grid_final_df[climate] = {}
    WRIA_writer[climate] = {}
    grid_writer[climate] = {}
    for group in groups:
      WRIA_final_df[climate][group] = {}
      grid_final_df[climate][group] = {}
      WRIA_writer[climate][group] = {}
      grid_writer[climate][group] = {}
      for period in periods:
        WRIA_writer[climate][group][period] = None
        grid_writer[climate][group][period] = None

  if read_from_aggregated_parquet == False:

    WRIA_writer = {}
    grid_writer = {}
    for climate in climates:
      WRIA_writer[climate] = {}
      grid_writer[climate] = {}
      for group in groups:
        WRIA_writer[climate][group] = {}
        grid_writer[climate][group] = {}
        for period in periods:
          WRIA_writer[climate][group][period] = None
          grid_writer[climate][group][period] = None

    for climate in climates:
      for group in groups:
        if group == 'ref':
          group_cols = ["WRIA_NR", "gcm", "scn","version","bc"]
          numeric_cols = ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
        elif group == 'annual':
          group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year"]
          numeric_cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                          'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                          'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                          'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                          'max_cold_spell_length', 'wsdi']
        elif group == 'monthly':
          group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year","doy_or_month"]
          numeric_cols = ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]

        #WRIA_results = {}
        #grid_results = {}
        #for period in periods:
        #  WRIA_results[period] = []
        #  grid_results[period] = []


        seperate = False
        for filename in os.listdir(droot): #each file represent various climate data source, gcm model, scn, and group (ref,ann,mon)
          WRIA_results = {}
          grid_results = {}
          for period in periods:
            WRIA_results[period] = []
            grid_results[period] = []

          full_path = os.path.join(droot, filename)
          if climate in filename and f'_{group}_' in filename and '.parquet' in filename and '_cesm2_' not in filename and '_mpi-esm1-2-hr_' not in filename:
            print(f'{climate} {group} {full_path}')
            df = downcast(pd.read_parquet(full_path))
            df = df[df["gcm"] != "ssp245"]  #a bug in previous process. Now remove these extra data.

            #to get WRIA mean
            wdf = df[df["gridid"].isin(vicwira_valid_gridids)]
            wdf = wdf.join(vicwira_area_indexed, on="gridid", how="left")

            WRIA_means = weighted_mean_vectorized(wdf, group_cols, numeric_cols, "km2")
            release_memory('wdf',scope=globals())  # free wdf immediately after use
            #WRIA_results.append(WRIA_means)
            if group == 'ref':
              WRIA_results['ref'].append(WRIA_means)
            elif group == 'annual':
              for period in [p for p in periods if p != 'ref']:
                pdf = WRIA_means[WRIA_means['year'].between(periods[period][0], periods[period][1])]
                pdf_means = pdf.groupby([x for x in group_cols if x not in ['year']], observed=True)[numeric_cols].mean().reset_index()
                WRIA_results[period].append(pdf_means)
                release_memory('pdf', 'pdf_means',scope=globals())
            elif group == 'monthly':
              for period in [p for p in periods if p != 'ref']:
                pdf = WRIA_means[WRIA_means['year'].between(periods[period][0], periods[period][1])]
                pdf_means = pdf.groupby([x for x in group_cols if x not in ['year']], observed=True)[numeric_cols].mean().reset_index()
                WRIA_results[period].append(pdf_means)
                release_memory('pdf', 'pdf_means',scope=globals())
            release_memory('WRIA_means',scope=globals())  # free after all periods processed


            #to get grid mean
            if group == 'ref':
              grid_means = df.groupby([x for x in group_cols if x != 'WRIA_NR'] + ['gridid'], observed=True)[numeric_cols].mean().reset_index()
              grid_results['ref'].append(grid_means)
              release_memory('grid_means',scope=globals())
            elif group == 'annual':
              for period in [p for p in periods if p != 'ref']:
                pdf = df[df['year'].between(periods[period][0], periods[period][1])]
                grid_means = pdf.groupby([x for x in group_cols if x not in ['WRIA_NR','year']] + ['gridid'], observed=True)[numeric_cols].mean().reset_index()
                grid_results[period].append(grid_means)
                release_memory('pdf', 'grid_means',scope=globals())
            elif group == 'monthly':
              for period in [p for p in periods if p != 'ref']:
                pdf = df[df['year'].between(periods[period][0], periods[period][1])]
                grid_means = pdf.groupby([x for x in group_cols if x not in ['WRIA_NR','year']] + ['gridid'], observed=True)[numeric_cols].mean().reset_index()
                grid_results[period].append(grid_means)
                release_memory('pdf', 'grid_means',scope=globals())

            release_memory('df',scope=globals())

            for period in periods:
              if len(WRIA_results[period]) > 0:
                print(f'WRIA: {climate} {group} {period}')
                tmpdf = pd.concat(WRIA_results[period]).reset_index(drop=True)
                table = pa.Table.from_pandas(tmpdf)
                if WRIA_writer[climate][group][period] is None:
                  WRIA_writer[climate][group][period] = pq.ParquetWriter(f'{outroot}/WRIA_{climate}_{group}_{period}_nocesm2_mpi-esm1-2-hr.parquet', table.schema)
                WRIA_writer[climate][group][period].write_table(table, row_group_size=50_000)
                WRIA_results[period] = []
                release_memory('tmpdf', 'table',scope=globals())

              if len(grid_results[period]) > 0:
                print(f'grid: {climate} {group} {period}')
                tmpdf = pd.concat(grid_results[period]).reset_index(drop=True)
                table = pa.Table.from_pandas(tmpdf)
                if grid_writer[climate][group][period] is None:
                  grid_writer[climate][group][period] = pq.ParquetWriter(f'{outroot}/grid_{climate}_{group}_{period}_nocesm2_mpi-esm1-2-hr.parquet', table.schema)
                grid_writer[climate][group][period].write_table(table, row_group_size=50_000)
                grid_results[period] = []
                release_memory('tmpdf', 'table',scope=globals())
            release_memory('WRIA_results', 'grid_results', scope=globals())
            gc.collect()
            check_mem_and_warn()
        for period in periods:
          if WRIA_writer[climate][group][period]:
            WRIA_writer[climate][group][period].close()
            WRIA_writer[climate][group][period] = None
          if grid_writer[climate][group][period]:
            grid_writer[climate][group][period].close()
            grid_writer[climate][group][period] = None
        for period in periods:
          fparquet = f'{outroot}/WRIA_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            WRIA_final_df[climate][group][period] = pd.read_parquet(fparquet)
          fparquet = f'{outroot}/grid_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            grid_final_df[climate][group][period] = pd.read_parquet(fparquet)


    gc.collect()

In [ ]:
#FOR ALL "maca3_cmip6" AND ""monthly"", SEPERATE EACH VARIABLE

#sgcm = 'mpi-esm1-2-hr'
if False:
  for var in ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]:
    climates = ["maca3_cmip6"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6"]
    groups = ["monthly"] #["ref","annual","monthly"]

    read_from_aggregated_parquet = False
    #do_cesm2_monthly = True #too big!

    WRIA_final_df = {}  #WRIA mean
    grid_final_df = {}  #grid level mean #1990s: 1981-2010 2030–2059 and 2070–2099 #only for annual and month var group
    for climate in climates:
      WRIA_final_df[climate] = {}
      grid_final_df[climate] = {}
      for group in groups:
        WRIA_final_df[climate][group] = {}
        grid_final_df[climate][group] = {}
    if read_from_aggregated_parquet == False:
      for climate in climates:
        for group in groups:
          if group == 'ref':
            group_cols = ["WRIA_NR", "gcm", "scn","version","bc"]
            numeric_cols = ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
          elif group == 'annual':
            group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year"]
            numeric_cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                            'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                            'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                            'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                            'max_cold_spell_length', 'wsdi']
          elif group == 'monthly':
            group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year","doy_or_month"]
            numeric_cols = ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]

          #WRIA_results = {}
          #grid_results = {}
          #for period in periods:
          #  WRIA_results[period] = []
          #  grid_results[period] = []


          seperate = False
          for filename in os.listdir(droot): #each file represent various climate data source, gcm model, scn, and group (ref,ann,mon)
            WRIA_results = {}
            grid_results = {}
            for period in periods:
              WRIA_results[period] = []
              grid_results[period] = []

            full_path = os.path.join(droot, filename)
            if climate in filename and f'_{group}_' in filename and '.parquet' in filename:
              sgcm = filename.split('_')[-2]
              print(f'{sgcm}::{climate} {group} {full_path}')
              df = downcast(pd.read_parquet(full_path,columns=["gridid","gcm", "scn","version","bc", "year","doy_or_month",var]))
              df = df[df["gcm"] != "ssp245"]  #a bug in previous process. Now remove these extra data.

              #to get WRIA mean
              wdf = df[df["gridid"].isin(vicwira_valid_gridids)]
              wdf = wdf.join(vicwira_area_indexed, on="gridid", how="left")

              WRIA_means = weighted_mean_vectorized(wdf, group_cols, [var], "km2")
              release_memory('wdf',scope=globals())  # free wdf immediately after use
              #WRIA_results.append(WRIA_means)
              if group == 'monthly':
                for period in [p for p in periods if p != 'ref']:
                  pdf = WRIA_means[WRIA_means['year'].between(periods[period][0], periods[period][1])]
                  pdf_means = pdf.groupby([x for x in group_cols if x not in ['year']], observed=True)[[var]].mean().reset_index()
                  WRIA_results[period].append(pdf_means)
                  release_memory('pdf', 'pdf_means',scope=globals())
              release_memory('WRIA_means',scope=globals())  # free after all periods processed


              #to get grid mean
              if group == 'monthly':
                for period in [p for p in periods if p != 'ref']:
                  pdf = df[df['year'].between(periods[period][0], periods[period][1])]
                  grid_means = pdf.groupby([x for x in group_cols if x not in ['WRIA_NR','year']] + ['gridid'], observed=True)[[var]].mean().reset_index()
                  grid_results[period].append(grid_means)
                  release_memory('pdf', 'grid_means',scope=globals())

              release_memory('df',scope=globals())

              for period in periods:
                if len(WRIA_results[period]) > 0:
                  print(f'WRIA: {climate} {group} {period}')
                  tmpdf = pd.concat(WRIA_results[period]).reset_index(drop=True)
                  #table = pa.Table.from_pandas(tmpdf)
                  #if WRIA_writer[climate][group][period] is None:
                  tmpdf.to_parquet(f'{outroot}/WRIA_{climate}_{group}_{period}_{sgcm}_{var}.parquet')
                  #WRIA_writer[climate][group][period].write_table(table, row_group_size=50_000)
                  #WRIA_results[period] = []
                  release_memory('tmpdf',scope=globals())

                if len(grid_results[period]) > 0:
                  print(f'grid: {climate} {group} {period}')
                  tmpdf = pd.concat(grid_results[period]).reset_index(drop=True)
                  #table = pa.Table.from_pandas(tmpdf)
                  #if grid_writer[climate][group][period] is None:
                  tmpdf.to_parquet(f'{outroot}/grid_{climate}_{group}_{period}_{sgcm}_{var}.parquet')
                  #grid_writer[climate][group][period].write_table(table, row_group_size=50_000)
                  #grid_results[period] = []
                  release_memory('tmpdf',scope=globals())
              release_memory('WRIA_results', 'grid_results', scope=globals())
              gc.collect()
              check_mem_and_warn()
      gc.collect()


In [ ]:
#merge maca3_smpi6  monthly results
if False:
  pwria = []
  pgrid = []
  for climate in ["maca3_cmip6"]:
    for group in ["monthly"]:
      for period in periods:
        pwria = {}
        pgrid = {}
        for var in ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]:
          twria = []
          tgrid = []
          for filename in os.listdir(outroot):
            print(f"{period} {filename}")
            if climate in filename and group in filename and period in filename and var in filename and '.parquet' in filename:
              print(f"{period} {filename}")
              if 'WRIA' in filename:
                t = pd.read_parquet(f'{outroot}/{filename}')
                twria.append(t)
              elif 'grid' in filename:
                t = pd.read_parquet(f'{outroot}/{filename}')
                tgrid.append(t)
          if len(twria) > 0:
            pwria[var] = pd.concat(twria, axis=0, ignore_index=True)
            pgrid[var] = pd.concat(tgrid, axis=0, ignore_index=True)
        if len(pwria) > 0:
          all_wira = pd.concat([df.set_index(["WRIA_NR", "gcm", "scn","version","bc", "doy_or_month"]) for df in [pwria["mon_ppt"],pwria["mon_tmax"],pwria["mon_tmin"],pwria["mon_tavg"]]]
                              , axis=1).reset_index()
          all_grid = pd.concat([df.set_index(["gridid", "gcm", "scn","version","bc", "doy_or_month"]) for df in [pgrid["mon_ppt"],pgrid["mon_tmax"],pgrid["mon_tmin"],pgrid["mon_tavg"]]]
                              , axis=1).reset_index()
          all_wira.to_parquet(f'{outroot}/WRIA_{climate}_{group}_{period}.parquet', index=False)
          all_grid.to_parquet(f'{outroot}/grid_{climate}_{group}_{period}.parquet', index=False)

In [ ]:
#merge WRF monthly results (special cases for two GCMs)
if False:
  pwria = []
  pgrid = []
  for climate in ["WRF_CMIP6"]:
    for group in ["monthly"]:
      for period in [p for p in periods if p != 'ref']:

        twria = []
        tgrid = []
        for gcm in ['cesm2','mpi-esm1-2-hr']:
          pWRIA = []
          pGRID = []
          for var in ['ppt','tmax','tmin','tavg']:
            fparquet = f'{outroot}/WRIA_WRF_CMIP6_monthly_{period}_{gcm}_mon_{var}.parquet'
            if os.path.exists(fparquet):
              print(f'pWRIA Append:{fparquet}')
              df = pd.read_parquet(fparquet)
              pWRIA.append(df)
            else:
              print(f'{fparquet} not exist!')
            fparquet = f'{outroot}/grid_WRF_CMIP6_monthly_{period}_{gcm}_mon_{var}.parquet'
            if os.path.exists(fparquet):
              print(f'pGRID Append:{fparquet}')
              df = pd.read_parquet(fparquet)
              pGRID.append(df)
            else:
              print(f'{fparquet} not exist!')
          if len(pWRIA) > 0:
            t = pd.concat(pWRIA, axis=1)
            t = t.loc[:, ~t.columns.duplicated()]
            twria.append(t)

            t = pd.concat(pGRID, axis=1)
            t = t.loc[:, ~t.columns.duplicated()]
            tgrid.append(t)
        if len(twria) > 0:
          gcm2_wria = pd.concat(twria, axis=0).reset_index(drop=True)
          gcm2_grid = pd.concat(tgrid, axis=0).reset_index(drop=True)


        tt_wria = []
        tt_grid = []
        tt_wria.append(gcm2_wria)
        tt_grid.append(gcm2_grid)

        fparquet = f'{outroot}/WRIA_{climate}_{group}_{period}_nocesm2_mpi-esm1-2-hr.parquet'
        if os.path.exists(fparquet):
          df = pd.read_parquet(fparquet)
          tt_wria.append(df)

        fparquet = f'{outroot}/grid_{climate}_{group}_{period}_nocesm2_mpi-esm1-2-hr.parquet'
        if os.path.exists(fparquet):
          df = pd.read_parquet(fparquet)
          tt_grid.append(df)

        tmpwria = pd.concat(tt_wria, axis=0).reset_index(drop=True)
        tmpgrid = pd.concat(tt_grid, axis=0).reset_index(drop=True)

        tmpwria.to_parquet(f'{outroot}/WRIA_{climate}_{group}_{period}.parquet', index=False)
        tmpgrid.to_parquet(f'{outroot}/grid_{climate}_{group}_{period}.parquet', index=False)

In [ ]:
#gcm2_wria.columns
#['WRIA_NR', 'gcm', 'scn', 'version', 'bc', 'doy_or_month', 'mon_ppt',
#       'mon_tmax', 'mon_tmin', 'mon_tavg']

#gcm2_grid.columns
#'gcm', 'scn', 'version', 'bc', 'doy_or_month', 'gridid', 'mon_ppt',
#       'mon_tmax', 'mon_tmin', 'mon_tavg'

#tt_wria[0].columns
#['WRIA_NR', 'gcm', 'scn', 'version', 'bc', 'doy_or_month', 'mon_ppt',
#       'mon_tmax', 'mon_tmin', 'mon_tavg']

#tt_grid[0].columns
#['gcm', 'scn', 'version', 'bc', 'doy_or_month', 'gridid', 'mon_ppt',
#       'mon_tmax', 'mon_tmin', 'mon_tavg']

In [ ]:
#ALL OTHER CONDITION!
if False:
  climates = ["ORNL"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6","ORNL"]
  groups = ["ref","annual","monthly"]


  #periods = {'ref':[],
  #          '1980s':[1981,2010],
  #          '2040s':[2030,2059],
  #          '2080s':[2070,2099]}

  read_from_aggregated_parquet = False
  if read_from_aggregated_parquet == False:
    user_input = input("Generate new aggregated parquet? Type 'yes' to confirm, anything else to skip: ").strip().lower()
    if user_input != 'yes':
      read_from_aggregated_parquet = True

  #do_cesm2_monthly = True #too big!
  WRIA_final_df = {}  #WRIA mean
  grid_final_df = {}  #grid level mean #1990s: 1981-2010 2030–2059 and 2070–2099 #only for annual and month var group
  WRIA_writer = {}
  grid_writer = {}
  for climate in climates:
    WRIA_final_df[climate] = {}
    grid_final_df[climate] = {}
    WRIA_writer[climate] = {}
    grid_writer[climate] = {}
    for group in groups:
      WRIA_final_df[climate][group] = {}
      grid_final_df[climate][group] = {}
      WRIA_writer[climate][group] = {}
      grid_writer[climate][group] = {}
      for period in periods:
        WRIA_writer[climate][group][period] = None
        grid_writer[climate][group][period] = None

  if read_from_aggregated_parquet == False:

    WRIA_writer = {}
    grid_writer = {}
    for climate in climates:
      WRIA_writer[climate] = {}
      grid_writer[climate] = {}
      for group in groups:
        WRIA_writer[climate][group] = {}
        grid_writer[climate][group] = {}
        for period in periods:
          WRIA_writer[climate][group][period] = None
          grid_writer[climate][group][period] = None

    for climate in climates:
      groups = ["ref","annual","monthly"]
      if climate in ['WRF_CMIP6','maca3_cmip6']: #already processed earlier
        if "monthly" in groups:
          groups.remove("monthly")
      for group in groups:
        if group == 'ref':
          group_cols = ["WRIA_NR", "gcm", "scn","version","bc"]
          numeric_cols = ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
          if climate in ['ORNL']:
            numeric_cols.remove("RL20_2070_2099")
        elif group == 'annual':
          group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year"]
          numeric_cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                          'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                          'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                          'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                          'max_cold_spell_length', 'wsdi']
        elif group == 'monthly':
          group_cols = ["WRIA_NR", "gcm", "scn","version","bc", "year","doy_or_month"]
          numeric_cols = ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]

        #WRIA_results = {}
        #grid_results = {}
        #for period in periods:
        #  WRIA_results[period] = []
        #  grid_results[period] = []


        seperate = False
        for filename in os.listdir(droot): #each file represent various climate data source, gcm model, scn, and group (ref,ann,mon)
          WRIA_results = {}
          grid_results = {}
          for period in periods:
            WRIA_results[period] = []
            grid_results[period] = []

          full_path = os.path.join(droot, filename)
          if climate in filename and f'_{group}_' in filename and '.parquet' in filename:
            print(f'{climate} {group} {full_path}')
            df = downcast(pd.read_parquet(full_path))
            df = df[df["gcm"] != "ssp245"]  #a bug in previous process. Now remove these extra data.

            #to get WRIA mean
            wdf = df[df["gridid"].isin(vicwira_valid_gridids)]
            wdf = wdf.join(vicwira_area_indexed, on="gridid", how="left")

            WRIA_means = weighted_mean_vectorized(wdf, group_cols, numeric_cols, "km2")
            release_memory('wdf',scope=globals())  # free wdf immediately after use
            #WRIA_results.append(WRIA_means)
            if group == 'ref':
              WRIA_results['ref'].append(WRIA_means)
            elif group == 'annual':
              for period in [p for p in periods if p != 'ref']:
                pdf = WRIA_means[WRIA_means['year'].between(periods[period][0], periods[period][1])]
                pdf_means = pdf.groupby([x for x in group_cols if x not in ['year']], observed=True)[numeric_cols].mean().reset_index()
                WRIA_results[period].append(pdf_means)
                release_memory('pdf', 'pdf_means',scope=globals())
            elif group == 'monthly':
              for period in [p for p in periods if p != 'ref']:
                pdf = WRIA_means[WRIA_means['year'].between(periods[period][0], periods[period][1])]
                pdf_means = pdf.groupby([x for x in group_cols if x not in ['year']], observed=True)[numeric_cols].mean().reset_index()
                WRIA_results[period].append(pdf_means)
                release_memory('pdf', 'pdf_means',scope=globals())
            release_memory('WRIA_means',scope=globals())  # free after all periods processed


            #to get grid mean
            if group == 'ref':
              grid_means = df.groupby([x for x in group_cols if x != 'WRIA_NR'] + ['gridid'], observed=True)[numeric_cols].mean().reset_index()
              grid_results['ref'].append(grid_means)
              release_memory('grid_means',scope=globals())
            elif group == 'annual':
              for period in [p for p in periods if p != 'ref']:
                pdf = df[df['year'].between(periods[period][0], periods[period][1])]
                grid_means = pdf.groupby([x for x in group_cols if x not in ['WRIA_NR','year']] + ['gridid'], observed=True)[numeric_cols].mean().reset_index()
                grid_results[period].append(grid_means)
                release_memory('pdf', 'grid_means',scope=globals())
            elif group == 'monthly':
              for period in [p for p in periods if p != 'ref']:
                pdf = df[df['year'].between(periods[period][0], periods[period][1])]
                grid_means = pdf.groupby([x for x in group_cols if x not in ['WRIA_NR','year']] + ['gridid'], observed=True)[numeric_cols].mean().reset_index()
                grid_results[period].append(grid_means)
                release_memory('pdf', 'grid_means',scope=globals())

            release_memory('df',scope=globals())

            for period in periods:
              if len(WRIA_results[period]) > 0:
                print(f'WRIA: {climate} {group} {period}')
                tmpdf = pd.concat(WRIA_results[period]).reset_index(drop=True)
                table = pa.Table.from_pandas(tmpdf)
                if WRIA_writer[climate][group][period] is None:
                  WRIA_writer[climate][group][period] = pq.ParquetWriter(f'{outroot}/WRIA_{climate}_{group}_{period}.parquet', table.schema)
                WRIA_writer[climate][group][period].write_table(table, row_group_size=50_000)
                WRIA_results[period] = []
                release_memory('tmpdf', 'table',scope=globals())

              if len(grid_results[period]) > 0:
                print(f'grid: {climate} {group} {period}')
                tmpdf = pd.concat(grid_results[period]).reset_index(drop=True)
                table = pa.Table.from_pandas(tmpdf)
                if grid_writer[climate][group][period] is None:
                  grid_writer[climate][group][period] = pq.ParquetWriter(f'{outroot}/grid_{climate}_{group}_{period}.parquet', table.schema)
                grid_writer[climate][group][period].write_table(table, row_group_size=50_000)
                grid_results[period] = []
                release_memory('tmpdf', 'table',scope=globals())
            release_memory('WRIA_results', 'grid_results', scope=globals())
            gc.collect()
            check_mem_and_warn()
        for period in periods:
          if WRIA_writer[climate][group][period]:
            WRIA_writer[climate][group][period].close()
            WRIA_writer[climate][group][period] = None
          if grid_writer[climate][group][period]:
            grid_writer[climate][group][period].close()
            grid_writer[climate][group][period] = None
        for period in periods:
          fparquet = f'{outroot}/WRIA_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            WRIA_final_df[climate][group][period] = pd.read_parquet(fparquet)
          fparquet = f'{outroot}/grid_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            grid_final_df[climate][group][period] = pd.read_parquet(fparquet)


    gc.collect()
  else:
    for climate in climates:
      for group in groups:
        for period in periods:
          fparquet = f'{outroot}/WRIA_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            WRIA_final_df[climate][group][period] = pd.read_parquet(fparquet)

          fparquet = f'{outroot}/grid_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            grid_final_df[climate][group][period] = pd.read_parquet(fparquet)


In [ ]:
#Read aggregation files
#NOTE: ORNL has two 'version': Daymet & Livneh

if True:
  climates = ["ORNL"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6"]
  groups = ["ref","annual","monthly"]
  read_from_aggregated_parquet = True
  if read_from_aggregated_parquet:
    WRIA_final_df = {}  #WRIA mean
    grid_final_df = {}  #grid level mean #1990s: 1981-2010 2030–2059 and 2070–2099 #only for annual and month var group
    for climate in climates:
      WRIA_final_df[climate] = {}
      grid_final_df[climate] = {}
      for group in groups:
        WRIA_final_df[climate][group] = {}
        grid_final_df[climate][group] = {}

    for climate in climates:
      for group in groups:
        for period in periods:
          fparquet = f'{outroot}/WRIA_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            WRIA_final_df[climate][group][period] = pd.read_parquet(fparquet)
          fparquet = f'{outroot}/grid_{climate}_{group}_{period}.parquet'
          if os.path.exists(fparquet):
            grid_final_df[climate][group][period] = pd.read_parquet(fparquet)


          if period in WRIA_final_df[climate][group]:
            print(f'WRIA_final_df: {climate} {group} {period}') # \n{WRIA_final_df[climate][group][period].columns}\n')
          if period in grid_final_df[climate][group]:
            print(f'grid_final_df: {climate} {group} {period}') # \n{grid_final_df[climate][group][period].columns}\n')
            #print(grid_final_df[climate][group][period].head)

In [ ]:
grid_final_df[climate]['ref']['ref']

In [ ]:
#get WA mean values (from WRIA and area-weight)
numeric_cols_dict = {'ref': ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099'],
                     'annual': ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                            'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                            'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                            'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                            'max_cold_spell_length', 'wsdi'],
                     'monthly': ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]}
group_cols_dict = {'ref': ["gcm", "scn","bc","climate","period"],
                  'annual': ["gcm", "scn","bc", "climate","period"],
                  'monthly': ["gcm", "scn","bc", "doy_or_month","climate","period"]}
climates = ["ORNL"] #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6"]
groups = ["ref","annual","monthly"]

import warnings

if True:
  level_means = {}
  for level in ['WRIA','WA','grid']:
    level_means[level] = {}
    if level == 'grid':
      tdf = grid_final_df.copy()
    else:
      tdf = WRIA_final_df.copy()
    for group in groups:
      tgroups = []
      for climate in climates:
        for period in periods:
          if tdf.get(climate, {}).get(group, {}).get(period) is not None:
            t = tdf[climate][group][period].copy()
            t['climate'] = climate
            t['period'] = period
            if level in ['WA']:
              t = t.join(wira_area_km2, on="WRIA_NR", how="left")
            tgroups.append(t)
      with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        tstate = pd.concat(tgroups,ignore_index=True)
      #tstate = tstate.convert_dtypes()
      tstate['bc'] = tstate['bc'].astype(str)
      tstate.loc[tstate['climate'] == 'gridmet', 'bc'] = 'bc'
      tstate.loc[tstate['climate'] == 'maca_v2', 'bc'] = 'bc'
      tstate.loc[tstate['climate'] == 'ORNL', 'bc'] = 'bc'
      tstate.loc[tstate['climate'] == 'maca3_cmip6', 'bc'] = 'bc'
      if group == 'ref':
        if 'RL20_2070_2099' not in tstate.columns:
           tstate['RL20_2070_2099'] = None
      if level == 'WA':
        level_means[level][group] = weighted_mean_vectorized(tstate, group_cols_dict[group], numeric_cols_dict[group], "km2")
      elif level == 'WRIA':
        level_means[level][group] = tstate.groupby(group_cols_dict[group] + ['WRIA_NR'], observed=True)[numeric_cols_dict[group]].mean().reset_index()
      else:
        level_means[level][group] = tstate.groupby([x for x in group_cols_dict[group] if x != 'gcm'] + ['gridid'], observed=True)[numeric_cols_dict[group]].mean().reset_index() #get the mean among all gcms

In [ ]:
level_means['grid']['ref']

In [ ]:
t = level_means['WA']['annual']
t = t[t['period'] == '1980s']
#t.loc[t['climate'] == 'gridmet', 'bc'] = 'bc'
cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm', 'R20mm_TOT',
        'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day', 'SDII',
        'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax', 'avg_tmin',
        'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
        'max_cold_spell_length', 'wsdi']

t = t.groupby(['climate'])[cols].agg([
    ('median', 'median'),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/annul_1980s_distribution_bc.csv')

In [ ]:
t = level_means['WA']['ref']
t = t[t['period'] == 'ref']
t = t[t['bc'] == 'bc']
#t.loc[t['climate'] == 'gridmet', 'bc'] = 'bc'
t = t.drop(columns=['RL20_2030_2059', 'RL20_2070_2099'])
cols = ['p99wet','p95wet','RL20']
t = t.groupby(['climate'])[cols].agg([
    ('median', 'median'),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/ref_ref_distribution_bc.csv')

In [ ]:
level_means['WA']['monthly'].columns

In [ ]:
t = level_means['WA']['monthly']
t = t[t['bc'] == 'bc']

#t.loc[t['climate'] == 'gridmet', 'bc'] = 'bc'
#t = t.drop(columns=['RL20_2030_2059', 'RL20_2070_2099'])
t = t.groupby(['climate','scn','doy_or_month','period'])[['mon_ppt','mon_tmax', 'mon_tmin', 'mon_tavg']].agg([
    ('median', 'median'),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/WA_distribution_bc_monthly_change.csv')

In [ ]:
t.columns

In [ ]:
t = level_means['WRIA']['ref']
t = t[t['period'] == 'ref']
t = t[t['bc'] == 'bc']
t = t.drop(columns=['RL20_2030_2059', 'RL20_2070_2099'])
cols = ['p99wet','p95wet','RL20']
t = t.groupby(['climate','WRIA_NR'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/ref_ref_distribution_WRIA.csv')

In [ ]:
t = level_means['WA']['annual']
t = t[t['bc'] == 'bc']
cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm', 'R20mm_TOT',
        'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day', 'SDII',
        'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax', 'avg_tmin',
        'fwet','wsdi']
t = t.groupby(['climate','scn','period'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/annul_1980s_2040s_2080s_distribution_WA_median_range.csv')

In [ ]:
t = level_means['WRIA']['annual']
t = t[t['bc'] == 'bc']
t = t[t['climate'] == 'WRF_CMIP6']
cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm', 'R20mm_TOT',
        'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day', 'SDII',
        'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax', 'avg_tmin',
        'fwet','wsdi']
t = t.groupby(['climate','scn','period','WRIA_NR'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/annul_1980s_2040s_2080s_distribution_WRIA_median_range_WRF_CMIP6.csv')

In [ ]:
t = level_means['WRIA']['annual']
t = t[t['bc'] == 'bc']
cols = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm', 'R20mm_TOT',
        'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day', 'SDII',
        'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax', 'avg_tmin',
        'fwet','wsdi']
t = t.groupby(['climate','WRIA_NR'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/annul_1980s_distribution_WRIA_median_range.csv')

In [ ]:
t = level_means['WRIA']['annual']
t = t[t['bc'] == 'bc']

cols = ['R99pTOT', 'Rx1day', 'Rx5day', 'SDII']

t = t.groupby(['climate','WRIA_NR'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/annul_1980s_distribution_WRIA_median_range_R99pTOT_Rx1day_Rx5day_SDII.csv')

In [ ]:
t = level_means['WA']['ref']
t = t[t['period'] == 'ref']
t = t[t['bc'] == 'bc']
#t = t[t['scn'].isin(['ssp370','rcp45','rcp85'])]
cols = ['RL20','RL20_2030_2059', 'RL20_2070_2099']

t = t.groupby(['climate','scn'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/testdata/ref_ref_distribution_change_scn.csv')

In [ ]:
t = level_means['WRIA']['ref']
t = t[t['period'] == 'ref']
t = t[t['bc'] == 'bc']
#t = t[t['climate'] == 'WRF_CMIP6']
#t = t[t['scn'].isin(['ssp370'])]
cols = ['RL20','RL20_2030_2059', 'RL20_2070_2099']

t = t.groupby(['climate','scn','WRIA_NR'])[cols].agg([
    ('median', 'median'),
    ('IQR',    lambda x: x.quantile(0.75) - x.quantile(0.25)),
    ('range',  lambda x: x.max() - x.min())
])

# Flatten MultiIndex columns: e.g. 'CDD_median', 'CDD_q25', ...
t.columns = ['_'.join(col) for col in t.columns]
t.to_csv('/content/drive/MyDrive/CMIP6/ClimateModuleReportData/ref_ref_distribution_change_scn_WRIA.csv')

In [ ]:
#Plot WA results
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#from adjustText import adjust_text
if False:
  region = 'WA'
  for sgroup in ['ref', 'annual']:
    value_cols = numeric_cols_dict[sgroup] #['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
    df = level_means[region][sgroup].copy()

    climate_order = ['gridmet', 'maca_v2', 'maca3_cmip6', 'WRF_CMIP6']
    df['climate_order'] = pd.Categorical(df['climate'], categories=climate_order, ordered=True)
    df = df.sort_values(['climate_order','scn', 'bc', 'period']).drop(columns='climate_order')

    group_cols = ['climate','scn', 'bc', 'period']
    # Create a combined group label
    df['pgroup'] = df[group_cols].astype(str).agg(' | '.join, axis=1)

    global_group_order = df[df[value_cols].ne(0).any(axis=1)]['pgroup'].drop_duplicates().tolist()
    # --- Find x positions where climate group changes ---
    climates = [g.split(' | ')[0] for g in global_group_order]
    vline_positions = []
    for i in range(1, len(climates)):
        if climates[i] != climates[i-1]:
            vline_positions.append(i - 0.5)  # place line between groups

    fig, axes = plt.subplots(len(value_cols), 1, figsize=(10, 2 * len(value_cols)), sharex=True)

    for ax, col in zip(axes, value_cols):
        df_filtered = df[df[col] != 0]

        sns.violinplot(data=df_filtered, x='pgroup', y=col, ax=ax,
                      color='lightblue', order=global_group_order)

        for i, group in enumerate(global_group_order):
            group_data = df_filtered[df_filtered['pgroup'] == group][col]
            if len(group_data) == 0:
                continue  # skip groups with no data for this col
            median_val = group_data.median()
            ax.text(i, median_val, f'{median_val:.1f}',
                    ha='center', va='bottom', fontsize=8, color='red')
        # --- Add vertical lines ---
        for xpos in vline_positions:
            ax.axvline(x=xpos, color='black', linewidth=1.2, linestyle='--', alpha=0.7)
        if 'RL20' in col:
          ax.set_ylim(20, 140)  # set min=0, max=100
        ax.set_ylabel(col)
        ax.set_xlabel('')

    axes[-1].set_xlabel(' | '.join(group_cols), fontsize=10)
    plt.xticks(rotation=30, ha='right', fontsize=6)
    plt.tight_layout()
    plt.savefig(f'{outroot}/violin_plot_{sgroup}_{region}.png', dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
#Plot WA montly results
if True:
  import pandas as pd
  import matplotlib.pyplot as plt
  import seaborn as sns
  #from adjustText import adjust_text
  import matplotlib.patches as mpatches
  import numpy as np

  #["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]}
  #'monthly': ["gcm", "scn","bc", "doy_or_month","climate","period"]}
  value_cols = ['mon_ppt', 'mon_tmax', 'mon_tmin', 'mon_tavg']
  group_cols = ['climate', 'scn']
  month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

  region = 'WA'
  sgroup = 'monthly'
  # Compute global y-limits per variable across all periods
  df_all = level_means[region][sgroup].copy()
  df_all.loc[df_all['climate'] == 'gridmet', 'bc'] = 'bc'
  df_all = df_all[df_all['bc'] == 'bc']

  ylims = {}
  for col in value_cols:
      ylims[col] = (df_all[col].min(), df_all[col].max())

  for sgroup in ['monthly']:
    tperiod = {}
    for period in ["1980s","2040s","2080s"]:
      value_cols = numeric_cols_dict[sgroup]
      df = level_means[region][sgroup].copy()
      df.loc[df['climate'] == 'gridmet', 'bc'] = 'bc'
      df = df[(df['bc'] == 'bc') & (df['period'] == period)]
      tperiod[period] = df.copy()
      groups = df.groupby(group_cols)
      colors = plt.cm.tab10.colors
      group_keys = list(groups.groups.keys())
      print(period)
      if len(df) > 0:
        fig, axes = plt.subplots(len(value_cols), 1, figsize=(8, 3 * len(value_cols)), sharex=True)
        line_styles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']

        for ax, col in zip(axes, value_cols):
            for idx, key in enumerate(group_keys):
                group_df = groups.get_group(key)
                color = colors[idx % len(colors)]
                linestyle = line_styles[idx % len(line_styles)]

                agg = group_df.groupby('doy_or_month')[col].agg(
                    median='median',
                    q25=lambda x: x.quantile(0.25),
                    q75=lambda x: x.quantile(0.75),
                    min='min',
                    max='max'
                ).reset_index().sort_values('doy_or_month')

                x = agg['doy_or_month']
                ax.fill_between(x, agg['min'], agg['max'], alpha=0.15, color=color)
                ax.fill_between(x, agg['q25'], agg['q75'], alpha=0.3, color=color)
                darker_color = tuple(c * 0.6 for c in color[:3])
                ax.plot(x, agg['median'], color=darker_color, linewidth=1.5, linestyle=linestyle)

            # Apply consistent y-limits
            ax.set_ylim(ylims[col])

            legend_patches = [
                plt.Line2D([0], [0],
                          color=tuple(c * 0.6 for c in colors[i % len(colors)][:3]),
                          linewidth=1.5,
                          linestyle=line_styles[i % len(line_styles)],
                          label=' | '.join(str(k) for k in group_keys[i]))
                for i in range(len(group_keys))
            ]
            ax.legend(handles=legend_patches, fontsize=7,
                      bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
            ax.set_ylabel(col)
            ax.set_title(period, loc='left', fontsize=9)
            ax.set_xlabel('')

        axes[-1].set_xticks(range(1, 13))
        axes[-1].set_xticklabels(month_labels)
        axes[-1].set_xlabel('Month')
        plt.tight_layout()
        plt.savefig(f'{outroot}/WA_monthly_{region}_{period}.png', dpi=600, bbox_inches='tight')
        plt.show()



In [ ]:
# Set index for both dataframes
#Plot monthly change
if False:
  df2 = tperiod['2040s']
  df1 = tperiod['1980s']
  index_cols = ['gcm', 'scn', 'bc', 'doy_or_month', 'climate']
  value_cols = ['mon_ppt', 'mon_tmax', 'mon_tmin', 'mon_tavg']

  df1_indexed = df1.set_index(index_cols)
  df2_indexed = df2.set_index(index_cols)
  # Calculate difference (df2 - df1), only on matching rows
  diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
  # Drop rows where df2 had no data (e.g. missing 'gridmat' climate rows)
  diff = diff.dropna(how='all')
  # Reset index if needed
  diff2040s_1980s = diff.reset_index()

  df2 = tperiod['2080s']
  df1_indexed = df1.set_index(index_cols)
  df2_indexed = df2.set_index(index_cols)
  # Calculate difference (df2 - df1), only on matching rows
  diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
  # Drop rows where df2 had no data (e.g. missing 'gridmat' climate rows)
  diff = diff.dropna(how='all')
  # Reset index if needed
  diff2080s_1980s = diff.reset_index()

  if True:
    tt = {}
    for period in ["2040s","2080s"]:
      tt[period] = {}
      for col in value_cols:
        if period == "2040s":
          df = diff2040s_1980s
        else:
          df = diff2080s_1980s
        tt[period][col] = (df[col].min(), df[col].max())
    ylims = {}
    for col in value_cols:
      t_min = min(tt["2040s"][col][0], tt["2080s"][col][0])
      t_max = max(tt["2040s"][col][1], tt["2080s"][col][1])
      ylims[col] = (t_min,t_max)


    for period in ["2040s","2080s"]:
      value_cols = numeric_cols_dict[sgroup]
      if period == "2040s":
        df = diff2040s_1980s
      else:
        df = diff2080s_1980s
      groups = df.groupby(group_cols)
      colors = plt.cm.tab10.colors
      group_keys = list(groups.groups.keys())
      print(period)
      if len(df) > 0:
        fig, axes = plt.subplots(len(value_cols), 1, figsize=(8, 3 * len(value_cols)), sharex=True)
        line_styles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']

        for ax, col in zip(axes, value_cols):
            for idx, key in enumerate(group_keys):
                group_df = groups.get_group(key)
                color = colors[idx % len(colors)]
                linestyle = line_styles[idx % len(line_styles)]

                agg = group_df.groupby('doy_or_month')[col].agg(
                    median='median',
                    q25=lambda x: x.quantile(0.25),
                    q75=lambda x: x.quantile(0.75),
                    min='min',
                    max='max'
                ).reset_index().sort_values('doy_or_month')

                x = agg['doy_or_month']
                ax.fill_between(x, agg['min'], agg['max'], alpha=0.15, color=color)
                ax.fill_between(x, agg['q25'], agg['q75'], alpha=0.3, color=color)
                darker_color = tuple(c * 0.6 for c in color[:3])
                ax.plot(x, agg['median'], color=darker_color, linewidth=1.5, linestyle=linestyle)

            # Apply consistent y-limits
            ax.set_ylim(ylims[col])

            legend_patches = [
                plt.Line2D([0], [0],
                          color=tuple(c * 0.6 for c in colors[i % len(colors)][:3]),
                          linewidth=1.5,
                          linestyle=line_styles[i % len(line_styles)],
                          label=' | '.join(str(k) for k in group_keys[i]))
                for i in range(len(group_keys))
            ]
            ax.legend(handles=legend_patches, fontsize=7,
                      bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0)
            ax.set_ylabel(col)
            ax.set_title(f'{period}-1980s', loc='left', fontsize=9)
            ax.set_xlabel('')

        axes[-1].set_xticks(range(1, 13))
        axes[-1].set_xticklabels(month_labels)
        axes[-1].set_xlabel('Month')
        plt.tight_layout()
        plt.savefig(f'{outroot}/WA_monthly_{region}_change_{period}_1980s.png', dpi=600, bbox_inches='tight')
        plt.show()

In [ ]:
# Set index for both dataframes
#Plot monthly change (smothting the curve)
import numpy as np
from scipy.interpolate import make_interp_spline
def smooth_monthly(x, y, wrap=True, n_points=300):
    """Smooth monthly data with wrap-around for Jan/Dec continuity."""
    x = np.array(x)
    y = np.array(y)

    if wrap:
        # Pad with Dec on left and Jan on right to smooth endpoints
        x_pad = np.concatenate([[x[-1] - 12], x, [x[0] + 12]])
        y_pad = np.concatenate([[y[-1]], y, [y[0]]])
    else:
        x_pad, y_pad = x, y

    spline = make_interp_spline(x_pad, y_pad, k=3)
    x_smooth = np.linspace(x[0], x[-1], n_points)
    y_smooth = spline(x_smooth)
    return x_smooth, y_smooth

if True:
  df2 = tperiod['2040s']
  df1 = tperiod['1980s']
  index_cols = ['gcm', 'scn', 'bc', 'doy_or_month', 'climate']
  value_cols = ['mon_ppt', 'mon_tmax', 'mon_tmin', 'mon_tavg']

  df1_indexed = df1.set_index(index_cols)
  df2_indexed = df2.set_index(index_cols)
  # Calculate difference (df2 - df1), only on matching rows
  diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
  # Drop rows where df2 had no data (e.g. missing 'gridmat' climate rows)
  diff = diff.dropna(how='all')
  # Reset index if needed
  diff2040s_1980s = diff.reset_index()

  df2 = tperiod['2080s']
  df1_indexed = df1.set_index(index_cols)
  df2_indexed = df2.set_index(index_cols)
  # Calculate difference (df2 - df1), only on matching rows
  diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
  # Drop rows where df2 had no data (e.g. missing 'gridmat' climate rows)
  diff = diff.dropna(how='all')
  # Reset index if needed
  diff2080s_1980s = diff.reset_index()

  if True:
    tt = {}
    for period in ["2040s","2080s"]:
      tt[period] = {}
      for col in value_cols:
        if period == "2040s":
          df = diff2040s_1980s
        else:
          df = diff2080s_1980s
        tt[period][col] = (df[col].min(), df[col].max())
    ylims = {}
    for col in value_cols:
      t_min = min(tt["2040s"][col][0], tt["2080s"][col][0])
      t_max = max(tt["2040s"][col][1], tt["2080s"][col][1])
      if col in ['mon_tmax', 'mon_tmin', 'mon_tavg']:
        ylims[col] = (t_min,t_max + 1)
      else:
        ylims[col] = (t_min,t_max)


    for period in ["2040s","2080s"]:
      value_cols = numeric_cols_dict[sgroup]
      if period == "2040s":
        df = diff2040s_1980s
      else:
        df = diff2080s_1980s
      groups = df.groupby(group_cols)
      colors = plt.cm.tab10.colors
      group_keys = list(groups.groups.keys())
      print(period)
      if len(df) > 0:
        fig, axes = plt.subplots(len(value_cols), 1, figsize=(8, 3 * len(value_cols)), sharex=True)
        line_styles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']

        for ax, col in zip(axes, value_cols):
            for idx, key in enumerate(group_keys):
                group_df = groups.get_group(key)
                color = colors[idx % len(colors)]
                linestyle = line_styles[idx % len(line_styles)]

                agg = group_df.groupby('doy_or_month')[col].agg(
                    median='median',
                    q25=lambda x: x.quantile(0.25),
                    q75=lambda x: x.quantile(0.75),
                    min='min',
                    max='max'
                ).reset_index().sort_values('doy_or_month')

                x = agg['doy_or_month'].values

                # Compute all smoothed curves
                x_s,   med_s = smooth_monthly(x, agg['median'].values)
                _,     q25_s = smooth_monthly(x, agg['q25'].values)
                _,     q75_s = smooth_monthly(x, agg['q75'].values)
                _,     min_s = smooth_monthly(x, agg['min'].values)
                _,     max_s = smooth_monthly(x, agg['max'].values)

                # Plot directly from smooth arrays
                ax.fill_between(x_s, min_s, max_s, alpha=0.15, color=color)
                ax.fill_between(x_s, q25_s, q75_s, alpha=0.3,  color=color)
                darker_color = tuple(c * 0.6 for c in color[:3])
                ax.plot(x_s, med_s, color=darker_color, linewidth=1.5, linestyle=linestyle)


            # Apply consistent y-limits
            if period == "2040s" and col in ['mon_tmax', 'mon_tmin', 'mon_tavg']:
              ax.set_ylim(-1,8)
            else:
              ax.set_ylim(ylims[col])

            if col == value_cols[0]:
              legend_patches = [
                  plt.Line2D([0], [0],
                            color=tuple(c * 0.6 for c in colors[i % len(colors)][:3]),
                            linewidth=1.5,
                            linestyle=line_styles[i % len(line_styles)],
                            label=' | '.join(str(k) for k in group_keys[i]))
                  for i in range(len(group_keys))
              ]
              ax.legend(handles=legend_patches, fontsize=7,
                        loc='upper center')
            ax.set_ylabel(col)
            ax.set_title(f'{period}-1980s', loc='left', fontsize=9)
            ax.set_xlabel('')

        axes[-1].set_xticks(range(1, 13))
        axes[-1].set_xticklabels(month_labels)
        axes[-1].set_xlabel('Month')
        plt.tight_layout()
        plt.savefig(f'{outroot}/WA_monthly_{region}_change_{period}_1980s_smothed.png', dpi=600, bbox_inches='tight')
        plt.show()

In [ ]:
# Set index for both dataframes
#Plot monthly change (smothting the curve) keep maca2 and WRF only
import numpy as np
from scipy.interpolate import make_interp_spline
def smooth_monthly(x, y, wrap=True, n_points=300):
    """Smooth monthly data with wrap-around for Jan/Dec continuity."""
    x = np.array(x)
    y = np.array(y)

    if wrap:
        # Pad with Dec on left and Jan on right to smooth endpoints
        x_pad = np.concatenate([[x[-1] - 12], x, [x[0] + 12]])
        y_pad = np.concatenate([[y[-1]], y, [y[0]]])
    else:
        x_pad, y_pad = x, y

    spline = make_interp_spline(x_pad, y_pad, k=3)
    x_smooth = np.linspace(x[0], x[-1], n_points)
    y_smooth = spline(x_smooth)
    return x_smooth, y_smooth

if False:
  df2 = tperiod['2040s'].copy()
  df1 = tperiod['1980s'].copy()
  df2 = df2[(df2['climate'] == 'maca_v2') | (df2['climate'] == 'WRF_CMIP6')]
  df1 = df1[(df1['climate'] == 'maca_v2') | (df1['climate'] == 'WRF_CMIP6') | (df2['climate'] == 'gridmet')]
  #['gridmet', 'maca_v2', 'maca3_cmip6', 'WRF_CMIP6']
  index_cols = ['gcm', 'scn', 'bc', 'doy_or_month', 'climate']
  value_cols = ['mon_ppt', 'mon_tmax', 'mon_tmin', 'mon_tavg']

  df1_indexed = df1.set_index(index_cols)
  df2_indexed = df2.set_index(index_cols)
  # Calculate difference (df2 - df1), only on matching rows
  diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
  # Drop rows where df2 had no data (e.g. missing 'gridmat' climate rows)
  diff = diff.dropna(how='all')
  # Reset index if needed
  diff2040s_1980s = diff.reset_index()

  df2 = tperiod['2080s']
  df1_indexed = df1.set_index(index_cols)
  df2_indexed = df2.set_index(index_cols)
  # Calculate difference (df2 - df1), only on matching rows
  diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
  # Drop rows where df2 had no data (e.g. missing 'gridmat' climate rows)
  diff = diff.dropna(how='all')
  # Reset index if needed
  diff2080s_1980s = diff.reset_index()

  if True:
    tt = {}
    for period in ["2040s","2080s"]:
      tt[period] = {}
      for col in value_cols:
        if period == "2040s":
          df = diff2040s_1980s
        else:
          df = diff2080s_1980s
        tt[period][col] = (df[col].min(), df[col].max())
    ylims = {}
    for col in value_cols:
      t_min = min(tt["2040s"][col][0], tt["2080s"][col][0])
      t_max = max(tt["2040s"][col][1], tt["2080s"][col][1])
      if col in ['mon_tmax', 'mon_tmin', 'mon_tavg']:
        ylims[col] = (t_min,t_max + 1)
      else:
        ylims[col] = (t_min,t_max)


    for period in ["2040s","2080s"]:
      value_cols = numeric_cols_dict[sgroup]
      if period == "2040s":
        df = diff2040s_1980s
      else:
        df = diff2080s_1980s
      groups = df.groupby(group_cols)
      colors = plt.cm.tab10.colors
      group_keys = list(groups.groups.keys())
      print(period)
      if len(df) > 0:
        fig, axes = plt.subplots(len(value_cols), 1, figsize=(8, 3 * len(value_cols)), sharex=True)
        line_styles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']

        for ax, col in zip(axes, value_cols):
            for idx, key in enumerate(group_keys):
                group_df = groups.get_group(key)
                color = colors[idx % len(colors)]
                linestyle = line_styles[idx % len(line_styles)]

                agg = group_df.groupby('doy_or_month')[col].agg(
                    median='median',
                    q25=lambda x: x.quantile(0.25),
                    q75=lambda x: x.quantile(0.75),
                    min='min',
                    max='max'
                ).reset_index().sort_values('doy_or_month')

                x = agg['doy_or_month'].values

                # Compute all smoothed curves
                x_s,   med_s = smooth_monthly(x, agg['median'].values)
                _,     q25_s = smooth_monthly(x, agg['q25'].values)
                _,     q75_s = smooth_monthly(x, agg['q75'].values)
                _,     min_s = smooth_monthly(x, agg['min'].values)
                _,     max_s = smooth_monthly(x, agg['max'].values)

                # Plot directly from smooth arrays
                ax.fill_between(x_s, min_s, max_s, alpha=0.15, color=color)
                ax.fill_between(x_s, q25_s, q75_s, alpha=0.3,  color=color)
                darker_color = tuple(c * 0.6 for c in color[:3])
                ax.plot(x_s, med_s, color=darker_color, linewidth=1.5, linestyle=linestyle)


            # Apply consistent y-limits
            if period == "2040s" and col in ['mon_tmax', 'mon_tmin', 'mon_tavg']:
              ax.set_ylim(-1,8)
            else:
              ax.set_ylim(ylims[col])

            if col == value_cols[0]:
              legend_patches = [
                  plt.Line2D([0], [0],
                            color=tuple(c * 0.6 for c in colors[i % len(colors)][:3]),
                            linewidth=1.5,
                            linestyle=line_styles[i % len(line_styles)],
                            label=' | '.join(str(k) for k in group_keys[i]))
                  for i in range(len(group_keys))
              ]
              ax.legend(handles=legend_patches, fontsize=7,
                        loc='upper center')
            ax.set_ylabel(col)
            ax.set_title(f'{period}-1980s', loc='left', fontsize=9)
            ax.set_xlabel('')

        axes[-1].set_xticks(range(1, 13))
        axes[-1].set_xticklabels(month_labels)
        axes[-1].set_xlabel('Month')
        plt.tight_layout()
        plt.savefig(f'{outroot}/WA_monthly_{region}_change_{period}_1980s_smothed_maca2_and_wrf.png', dpi=600, bbox_inches='tight')
        plt.show()

In [ ]:
# Plot monthly change (smoothing the curve) keep maca2 and WRF only
# Water Year: Oct - Sep

import numpy as np
from scipy.interpolate import make_interp_spline

# --- Water-year helpers ---
wy_order = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
wy_pos   = {m: i+1 for i, m in enumerate(wy_order)}   # {10:1, 11:2, 12:3, 1:4, ...}
month_labels_wy = ['Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep']


def smooth_monthly(x, y, wrap=True, n_points=300):
    """Smooth monthly data with wrap-around for Oct/Sep continuity (water year)."""
    x = np.array(x)
    y = np.array(y)

    if wrap:
        # Pad with Sep on left and Oct on right to smooth endpoints
        x_pad = np.concatenate([[x[-1] - 12], x, [x[0] + 12]])
        y_pad = np.concatenate([[y[-1]], y, [y[0]]])
    else:
        x_pad, y_pad = x, y

    spline = make_interp_spline(x_pad, y_pad, k=3)
    x_smooth = np.linspace(x[0], x[-1], n_points)
    y_smooth = spline(x_smooth)
    return x_smooth, y_smooth


if True:
    df2 = tperiod['2040s'].copy()
    df1 = tperiod['1980s'].copy()
    df2 = df2[(df2['climate'] == 'maca_v2') | (df2['climate'] == 'WRF_CMIP6')]
    df1 = df1[(df1['climate'] == 'maca_v2') | (df1['climate'] == 'WRF_CMIP6') | (df2['climate'] == 'gridmet')]

    index_cols = ['gcm', 'scn', 'bc', 'doy_or_month', 'climate']
    value_cols = ['mon_ppt', 'mon_tmax', 'mon_tmin', 'mon_tavg']

    df1_indexed = df1.set_index(index_cols)
    df2_indexed = df2.set_index(index_cols)

    # Calculate difference (2040s - 1980s), only on matching rows
    diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
    diff = diff.dropna(how='all')
    diff2040s_1980s = diff.reset_index()

    df2 = tperiod['2080s']
    df2_indexed = df2.set_index(index_cols)

    # Calculate difference (2080s - 1980s), only on matching rows
    diff = df2_indexed[value_cols].subtract(df1_indexed[value_cols])
    diff = diff.dropna(how='all')
    diff2080s_1980s = diff.reset_index()

    if True:
        # Compute y-limits across both periods for consistent axes
        tt = {}
        for period in ["2040s", "2080s"]:
            tt[period] = {}
            for col in value_cols:
                df = diff2040s_1980s if period == "2040s" else diff2080s_1980s
                tt[period][col] = (df[col].min(), df[col].max())

        ylims = {}
        for col in value_cols:
            t_min = min(tt["2040s"][col][0], tt["2080s"][col][0])
            t_max = max(tt["2040s"][col][1], tt["2080s"][col][1])
            if col in ['mon_tmax', 'mon_tmin', 'mon_tavg']:
                ylims[col] = (t_min, t_max + 1)
            else:
                ylims[col] = (t_min, t_max)

        for period in ["2040s", "2080s"]:
            value_cols = numeric_cols_dict[sgroup]
            df = diff2040s_1980s if period == "2040s" else diff2080s_1980s

            # Remap months to water-year positions (Oct=1 ... Sep=12)
            df = df.copy()
            df['wy_pos'] = df['doy_or_month'].map(wy_pos)

            groups = df.groupby(group_cols)
            colors = plt.cm.tab10.colors
            group_keys = list(groups.groups.keys())
            print(period)

            if len(df) > 0:
                fig, axes = plt.subplots(len(value_cols), 1, figsize=(8, 3 * len(value_cols)), sharex=True)
                line_styles = ['-', '--', '-.', ':', '-', '--', '-.', ':', '-', '--']

                for ax, col in zip(axes, value_cols):
                    for idx, key in enumerate(group_keys):
                        group_df = groups.get_group(key)
                        color = colors[idx % len(colors)]
                        linestyle = line_styles[idx % len(line_styles)]

                        agg = group_df.groupby('wy_pos')[col].agg(
                            median='median',
                            q25=lambda x: x.quantile(0.25),
                            q75=lambda x: x.quantile(0.75),
                            min='min',
                            max='max'
                        ).reset_index().sort_values('wy_pos')   # sorted by WY order

                        x = agg['wy_pos'].values   # WY positions 1–12

                        # Compute all smoothed curves
                        x_s,  med_s = smooth_monthly(x, agg['median'].values)
                        _,    q25_s = smooth_monthly(x, agg['q25'].values)
                        _,    q75_s = smooth_monthly(x, agg['q75'].values)
                        _,    min_s = smooth_monthly(x, agg['min'].values)
                        _,    max_s = smooth_monthly(x, agg['max'].values)

                        # Plot
                        ax.fill_between(x_s, min_s, max_s, alpha=0.15, color=color)
                        ax.fill_between(x_s, q25_s, q75_s, alpha=0.3,  color=color)
                        darker_color = tuple(c * 0.6 for c in color[:3])
                        ax.plot(x_s, med_s, color=darker_color, linewidth=1.5, linestyle=linestyle)

                    # Apply consistent y-limits
                    if period == "2040s" and col in ['mon_tmax', 'mon_tmin', 'mon_tavg']:
                        ax.set_ylim(-1, 8)
                    else:
                        ax.set_ylim(ylims[col])

                    # Legend on first subplot only
                    if col == value_cols[0]:
                        legend_patches = [
                            plt.Line2D([0], [0],
                                       color=tuple(c * 0.6 for c in colors[i % len(colors)][:3]),
                                       linewidth=1.5,
                                       linestyle=line_styles[i % len(line_styles)],
                                       label=' | '.join(str(k) for k in group_keys[i]))
                            for i in range(len(group_keys))
                        ]
                        ax.legend(handles=legend_patches, fontsize=7, loc='upper center')

                    ax.set_ylabel(col)
                    ax.set_title(f'{period}-1980s', loc='left', fontsize=9)
                    ax.set_xlabel('')

                # Water-year x-axis: Oct–Sep
                axes[-1].set_xticks(range(1, 13))
                axes[-1].set_xticklabels(month_labels_wy)
                axes[-1].set_xlabel('Month (Water Year: Oct–Sep)')

                plt.tight_layout()
                plt.savefig(
                    f'{outroot}/WA_monthly_{region}_change_{period}_1980s_smoothed_maca2_and_wrf_WY.png',
                    dpi=600, bbox_inches='tight'
                )
                plt.show()

In [ ]:
#Plot WRIA level bars ('ref')
#level_means['WRIA']:'ref' 'annual' 'monthly'
#['gcm', 'scn', 'bc', 'climate', 'period', 'WRIA_NR', 'p99wet', 'p95wet',
#       'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
if False:
  variables = ['p99wet', 'p95wet','RL20']
  for group in ['ref']: #,'annual']:
    df = level_means['WRIA'][group].copy()
    df.loc[df['climate'] == 'gridmet', 'bc'] = 'bc'
    #df = df[(df['bc'] == 'bc') & (df['climate'].isin(['gridmet','maca3_cmip6','WRF_CMIP6','maca_v2'])) & (~df['scn'].isin(['ssp245','ssp585']))]
    df = df[(df['bc'] == 'bc') & (df['climate'].isin(['gridmet','WRF_CMIP6','maca3_cmip6','maca_v2'])) & (~df['scn'].isin(['ssp245','ssp585']))]
    df[['RL20_2030_2059', 'RL20_2070_2099']] = df[['RL20_2030_2059', 'RL20_2070_2099']].replace(0, float('nan'))

    fig, axes = plt.subplots(3, 1, figsize=(18, 15))

    for ax, var in zip(axes, variables):
        plot_df = df[['gcm', 'scn', 'climate', 'WRIA_NR', var]].copy()
        plot_df['pgroup'] = plot_df['climate'] + '_' + plot_df['scn']

        sns.boxplot(
            data=plot_df,
            x='WRIA_NR',
            y=var,
            hue='pgroup',
            ax=ax,
            showfliers=False
        )

        # Add vertical separators between WRIA_NR groups
        wria_values = sorted(plot_df['WRIA_NR'].unique())
        for i in range(1, len(wria_values)):
            ax.axvline(x=i - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        ax.set_title(var)
        ax.set_xlabel('WRIA_NR')
        ax.set_ylabel(var)
        ax.tick_params(axis='x', rotation=90)
        ax.legend(title='gcm / scn', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)

    plt.tight_layout()
    plt.savefig(f'{outroot}/WRIA_{group}.png', bbox_inches='tight')
    plt.show()

    #plot 'RL20', 'RL20_2030_2059', 'RL20_2070_2099'
    df = level_means['WRIA'][group].copy()
    df.loc[df['climate'] == 'gridmet', 'bc'] = 'bc'
    #df = df[(df['bc'] == 'bc') & (df['climate'].isin(['gridmet','maca3_cmip6','WRF_CMIP6','maca_v2'])) & (~df['scn'].isin(['ssp245','ssp585']))]
    df = df[(df['bc'] == 'bc') & (df['climate'].isin(['WRF_CMIP6'])) & (~df['scn'].isin(['ssp245','ssp585']))]
    df[['RL20_2030_2059', 'RL20_2070_2099']] = df[['RL20_2030_2059', 'RL20_2070_2099']].replace(0, float('nan'))

    fig, ax = plt.subplots(figsize=(18, 6))

    # Melt the three RL20 columns into long form
    plot_df = df[['climate', 'WRIA_NR', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']].copy()
    plot_df = plot_df.melt(
        id_vars=['climate', 'WRIA_NR'],
        value_vars=['RL20', 'RL20_2030_2059', 'RL20_2070_2099'],
        var_name='period',
        value_name='value'
    )

    # Combined x-axis: WRIA_NR, grouped by climate
    plot_df['x_group'] = plot_df['WRIA_NR'].astype(str) + '_' + plot_df['climate']

    sns.boxplot(
        data=plot_df,
        x='WRIA_NR',
        y='value',
        hue='period',
        ax=ax,
        showfliers=False
    )

    # Add vertical separators between WRIA_NR groups
    wria_values = sorted(df['WRIA_NR'].unique())
    n_climates = df['climate'].nunique()
    for i in range(1, len(wria_values)):
        ax.axvline(x=i * n_climates - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    ax.set_title('RL20 comparison across periods')
    ax.set_xlabel('WRIA_NR')
    ax.set_ylabel('RL20')
    ax.tick_params(axis='x', rotation=90)
    ax.legend(title='period', bbox_to_anchor=(1.01, 1), loc='upper left')

    plt.tight_layout()
    plt.savefig(f'{outroot}/WRIA_{group}_RL20_RL20_2030_2059_RL20_2070_2099_from_WRF.png', bbox_inches='tight')
    plt.show()

In [ ]:
#plot only RL20 for different period

#level_means['WRIA']:'ref' 'annual' 'monthly'
#['gcm', 'scn', 'bc', 'climate', 'period', 'WRIA_NR', 'p99wet', 'p95wet',
#       'RL20', 'RL20_2030_2059', 'RL20_2070_2099']
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#from adjustText import adjust_text
import matplotlib.patches as mpatches
import numpy as np
if True:
  variables = ['RL20']
  for group in ['ref']:
    df = level_means['WRIA'][group].copy()
    df.loc[df['climate'] == 'gridmet', 'bc'] = 'bc'
    df = df[(df['bc'] == 'bc') & (df['climate'].isin(['gridmet','WRF_CMIP6','maca3_cmip6','maca_v2'])) & (~df['scn'].isin(['ssp245','ssp585']))]
    df[['RL20_2030_2059', 'RL20_2070_2099']] = df[['RL20_2030_2059', 'RL20_2070_2099']].replace(0, float('nan'))

    # ── Combined figure with two equal-sized subplots ────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))  # equal height per subplot

    YMIN, YMAX = 0, 250  # shared y-axis limits

    # ── Plot 1 ────────────────────────────────────────────────────────────
    var = variables[0]
    plot_df = df[['gcm', 'scn', 'climate', 'WRIA_NR', var]].copy()
    plot_df['pgroup'] = plot_df['climate'] + '_' + plot_df['scn']

    sns.boxplot(
        data=plot_df,
        x='WRIA_NR',
        y=var,
        hue='pgroup',
        ax=ax1,
        showfliers=False
    )

    wria_values = sorted(plot_df['WRIA_NR'].unique())
    for i in range(1, len(wria_values)):
        ax1.axvline(x=i - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    ax1.set_ylim(YMIN, YMAX)
    ax1.set_title('NL20 during 1981-2010')
    ax1.set_xlabel('WRIA_NR')
    ax1.set_ylabel(var)
    ax1.tick_params(axis='x', rotation=90)
    # Plot 1 - change this line:
    ax1.legend(title='gcm / scn', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12)
    # To:
    ax1.legend(title='gcm / scn', loc='upper right', fontsize=12)

    # ── Plot 2 ────────────────────────────────────────────────────────────
    df2 = level_means['WRIA'][group].copy()
    df2.loc[df2['climate'] == 'gridmet', 'bc'] = 'bc'
    df2 = df2[(df2['bc'] == 'bc') & (df2['climate'].isin(['WRF_CMIP6'])) & (~df2['scn'].isin(['ssp245','ssp585']))]
    df2[['RL20_2030_2059', 'RL20_2070_2099']] = df2[['RL20_2030_2059', 'RL20_2070_2099']].replace(0, float('nan'))

    plot_df2 = df2[['climate', 'WRIA_NR', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099']].copy()
    plot_df2 = plot_df2.rename(columns={'RL20': 'RL20 (1981-2010)', 'RL20_2030_2059': 'RL20 (2030-2059)', 'RL20_2070_2099': 'RL20 (2070-2099)'})
    plot_df2 = plot_df2.melt(
        id_vars=['climate', 'WRIA_NR'],
        value_vars=['RL20 (1981-2010)', 'RL20 (2030-2059)', 'RL20 (2070-2099)'],
        var_name='period',
        value_name='value'
    )

    sns.boxplot(
        data=plot_df2,
        x='WRIA_NR',
        y='value',
        hue='period',
        ax=ax2,
        showfliers=False
    )

    wria_values2 = sorted(df2['WRIA_NR'].unique())
    n_climates = df2['climate'].nunique()
    for i in range(1, len(wria_values2)):
        ax2.axvline(x=i * n_climates - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    ax2.set_ylim(YMIN, YMAX)
    ax2.set_title('RL20 during various periods')
    ax2.set_xlabel('WRIA_NR')
    ax2.set_ylabel('RL20')
    ax2.tick_params(axis='x', rotation=90)
    # Plot 2 - change this line:
    ax2.legend(title='period', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12)
    # To:
    ax2.legend(title='period', loc='upper right', fontsize=12)

    plt.tight_layout()
    plt.savefig(f'{outroot}/WRIA_NL20_and_2040_2060_comapare_and_Change.png', bbox_inches='tight')
    plt.show()

In [ ]:
#Plot WRIA level bars ('annual') 1980s
if False:
  variables = ['CDD', 'CWD', 'FD',
        'GSL', 'ID', 'PRCPTOT', 'R20mm', 'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p',
        'R99pTOT', 'Rx1day', 'Rx5day', 'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt',
        'avg_tavg', 'avg_tmax', 'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p',
        'fwet', 'hwdi', 'max_cold_spell_length', 'wsdi']

  for group in ['annual']:
      df = level_means['WRIA'][group].copy()
      df.loc[df['climate'] == 'gridmet', 'bc'] = 'bc'
      df = df[(df['bc'] == 'bc') & (df['climate'].isin(['gridmet','WRF_CMIP6','maca3_cmip6','maca_v2'])) & (~df['scn'].isin(['ssp245','ssp585'])) & (df['period'] == '1980s')]

      n_vars = len(variables)
      fig, axes = plt.subplots(
          n_vars, 1,
          figsize=(20, 4 * n_vars),   # ← 4 inches per subplot
          sharex=True                  # ← shared x-axis
      )

      wria_values = sorted(df['WRIA_NR'].unique())

      for i, (ax, var) in enumerate(zip(axes, variables)):
          plot_df = df[['gcm', 'scn', 'climate', 'WRIA_NR', var]].copy()
          plot_df['pgroup'] = plot_df['climate'] + '_' + plot_df['scn']

          sns.boxplot(
              data=plot_df,
              x='WRIA_NR',
              y=var,
              hue='pgroup',
              ax=ax,
              showfliers=False,
              order=wria_values
          )

          # Vertical separators between WRIA_NR groups
          for j in range(1, len(wria_values)):
              ax.axvline(x=j - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

          ax.set_ylabel(var, fontsize=9)
          ax.set_xlabel('')  # hide x label for all but last
          ax.legend(title='climate / scn', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)

          # Only show x tick labels on the last subplot
          if i < n_vars - 1:
              ax.tick_params(labelbottom=False)
          else:
              ax.set_xlabel('WRIA_NR')
              ax.tick_params(axis='x', rotation=90)

      plt.tight_layout()
      plt.savefig(f'{outroot}/WRIA_{group}_1980s.png', bbox_inches='tight', dpi=300)
      plt.show()

In [ ]:
#plot WRIA annuals
if False:
  import matplotlib.pyplot as plt
  import seaborn as sns

  variables = ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm', 'R20mm_TOT',
              'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day', 'SDII',
              'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax', 'avg_tmin',
              'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
              'max_cold_spell_length', 'wsdi']
  for group in ['annual']:
    df = level_means['WRIA'][group].copy()
    df = df[(df['bc'] == 'bc') & (df['climate'].isin(['WRF_CMIP6'])) & (~df['scn'].isin(['ssp245','ssp585']))]

    # Filter to only the three periods of interest
    periods = ['1980s', '2040s', '2080s']

    df_filtered = df[df['period'].isin(periods)].copy()

    # Combined hue: period + scn
    df_filtered['period_scn'] = df_filtered['period']

    wria_values = sorted(df_filtered['WRIA_NR'].unique())
    n_vars = len(variables)

    fig, axes = plt.subplots(
        n_vars, 1,
        figsize=(22, 4 * n_vars),
        sharex=True
    )

    for i, (ax, var) in enumerate(zip(axes, variables)):
        plot_df = df_filtered[['scn', 'period', 'period_scn', 'WRIA_NR', var]].copy()

        sns.boxplot(
            data=plot_df,
            x='WRIA_NR',
            y=var,
            hue='period_scn',
            ax=ax,
            showfliers=False,
            order=wria_values,
        )

        # Vertical separators between WRIA_NR groups
        for j in range(1, len(wria_values)):
            ax.axvline(x=j - 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        ax.set_ylabel(var, fontsize=9)
        ax.set_xlabel('')
        ax.legend(title='period', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)

        if i < n_vars - 1:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel('WRIA_NR')
            ax.tick_params(axis='x', rotation=90)

    plt.tight_layout()
    plt.savefig(f'{outroot}/WRIA_{group}_1980s_2040s_2080s.png', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
def make_nice_ticks(vmax, n=8):
    raw = np.linspace(0, np.sqrt(vmax), n) ** 2
    return [round(v / 50) * 50 for v in raw]

ranges = {'PNW':{'lon':[-125, -107.5],'lat':[38.5, 49.5]},
          'WA':{'lon':[-125, -116.8],'lat':[45.2, 49.2]}
         }

ticks_and_color = {'RL20': {'tick_vals':make_nice_ticks(250),
                            'tick_text':make_nice_ticks(250)
                            },
                   'RL20_2030_2059': {'tick_vals':make_nice_ticks(250),
                            'tick_text':make_nice_ticks(250)
                            },
                   'RL20_2070_2099': {'tick_vals':make_nice_ticks(250),
                            'tick_text':make_nice_ticks(250)
                            },
                  'p95wet': {'tick_vals':make_nice_ticks(100),
                            'tick_text':make_nice_ticks(100)
                            },
                  'p99wet': {'tick_vals':make_nice_ticks(200),
                            'tick_text':make_nice_ticks(200)
                            },
                  'PRCPTOT': {'tick_vals':make_nice_ticks(2500),
                            'tick_text':make_nice_ticks(2500)
                            },
                  'SDII': {'tick_vals':make_nice_ticks(20),
                            'tick_text':make_nice_ticks(20)
                          },
                   'R95pTOT': {'tick_vals':make_nice_ticks(1000),
                            'tick_text':make_nice_ticks(1000)
                            },
                   'R99pTOT': {'tick_vals':make_nice_ticks(1000),
                            'tick_text':make_nice_ticks(1000)
                            },
                   'R20mm_TOT': {'tick_vals':make_nice_ticks(1000),
                            'tick_text':make_nice_ticks(1000)
                            },
                   'Rx1day': {'tick_vals':make_nice_ticks(200),
                            'tick_text':make_nice_ticks(200)
                            },
                   'Rx5day': {'tick_vals':make_nice_ticks(500),
                            'tick_text':make_nice_ticks(500)
                            },
                   'ann_ppt': {'tick_vals':make_nice_ticks(2500),
                            'tick_text':make_nice_ticks(2500)
                            },
                   'CDD': {'tick_vals':list(range(0,81,10)),
                            'tick_text':list(range(0,81,10))
                            },
                   'CWD': {'tick_vals':list(range(0,31,10)),
                            'tick_text':list(range(0,31,10))
                            },
                   'FD': {'tick_vals':list(range(0,201,20)),
                            'tick_text':list(range(0,201,20))
                            },
                   'GSL': {'tick_vals':list(range(60,366,30)),
                            'tick_text':list(range(60,366,30))
                            },
                   'ID': {'tick_vals':list(range(0,101,10)),
                            'tick_text':list(range(0,101,10))
                            },
                   'csdi': {'tick_vals':list(range(0,11,1)),
                            'tick_text':list(range(0,11,1))
                            },
                   'R95p': {'tick_vals':list(range(0,11,1)),
                            'tick_text':list(range(0,11,1))
                            },
                   'R99p': {'tick_vals':list(range(0,11,1)),
                            'tick_text':list(range(0,11,1))
                            },
                   'SU': {'tick_vals':list(range(0,201,30)),
                            'tick_text':list(range(0,201,30))
                            },
                   'days_TX90p': {'tick_vals':list(range(10,51,5)),
                            'tick_text':list(range(10,51,5))
                            },
                   'R20mm': {'tick_vals':list(range(0,101,10)),
                            'tick_text':list(range(0,101,10))
                            },
                   'max_cold_spell_length': {'tick_vals':list(range(0,11,1)),
                            'tick_text':list(range(0,11,1))
                            },
                   'TXx': {'tick_vals':list(range(25,45,2)),
                            'tick_text':list(range(25,45,2))
                            },
                   'TNn': {'tick_vals':list(range(-30,1,2)),
                            'tick_text':list(range(-30,1,2))
                            },
                   'avg_tavg': {'tick_vals':list(range(-5,25,5)),
                            'tick_text':list(range(-5,25,5))
                            },
                   'avg_tmax': {'tick_vals':list(range(0,31,3)),
                            'tick_text':list(range(0,31,3))
                            },
                   'avg_tmin': {'tick_vals':list(range(-10,21,3)),
                            'tick_text':list(range(-10,21,3))
                            },
                   'days_TN10p':{'tick_vals':list(range(0,51,2)),
                            'tick_text':list(range(0,51,2))
                            },
                   'days_TX90p':{'tick_vals':list(range(10,200,20)),
                            'tick_text':list(range(10,200,20))
                            },
                   'fwet':{'tick_vals':list(np.arange(0, 0.51, 0.05)),
                            'tick_text':[f'{x:.1f}' for x in np.arange(0, 0.51, 0.05)]
                            },
                   'wsdi':{'tick_vals':list(range(10,51,5)),
                            'tick_text':list(range(10,51,5))
                            },
                   'hwdi':{'tick_vals':list(range(0,16,2)),
                            'tick_text':list(range(0,16,2))
                            },
                  }

numeric_cols_dict = {'ref': ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099'],
                     'annual': ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                            'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                            'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                            'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                            'max_cold_spell_length', 'wsdi'],
                     'monthly': ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]}
group_cols_dict = {'ref': ["gcm", "scn","bc","climate","period"],
                  'annual': ["gcm", "scn","bc", "climate","period"],
                  'monthly': ["gcm", "scn","bc", "doy_or_month","climate","period"]}

In [ ]:
#level_means['grid'] #['ref']['annual']
#plot grid level mean
states = gpd.read_file('https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json')
wiras = gpd.read_file(f'{droot}/ECY_-2366300735965880701.geojson')
if True:
  numeric_cols_dict = {'ref': ['p99wet', 'p95wet', 'RL20', 'RL20_2030_2059', 'RL20_2070_2099'],
                      'annual': ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'PRCPTOT', 'R20mm',
                              'R20mm_TOT', 'R95p', 'R95pTOT', 'R99p', 'R99pTOT', 'Rx1day', 'Rx5day',
                              'SDII', 'SU', 'TNn', 'TXx', 'ann_ppt', 'avg_tavg', 'avg_tmax',
                              'avg_tmin', 'csdi', 'days_TN10p', 'days_TX90p', 'fwet', 'hwdi',
                              'max_cold_spell_length', 'wsdi'],
                      'monthly': ["mon_ppt","mon_tmax","mon_tmin","mon_tavg"]}
  group_cols_dict = {'ref': ["gcm", "scn","bc","climate","period"],
                    'annual': ["gcm", "scn","bc", "climate","period"],
                    'monthly': ["gcm", "scn","bc", "doy_or_month","climate","period"]}





  map_region = 'WA' #'PNW' #'WA'
  #scn = 'ssp370' #'ssp370'   #['obs', 'rcp45', 'rcp85', 'ssp245', 'ssp370', 'ssp585']
  bc = 'bc' # 'bc'

  for climate in ["ORNL"]: #["gridmet","maca_v2","maca3_cmip6","WRF_CMIP6"]:
    for scn in ['obs', 'rcp45', 'rcp85', 'ssp245', 'ssp370', 'ssp585']:
      for group in ["ref","annual"]:
        for period in ['ref','2000s','2040s','2080s']:
          for pvar in numeric_cols_dict[group]:
            t = level_means['grid'][group].copy()
            t.loc[t['climate'] == 'gridmet', 'bc'] = 'bc'
            t = t[t['climate'] == climate]
            t = t[t['period'] == period]
            if scn == None and bc == None:
              pass
            elif scn == None:
              t = t[t['bc'] == 'bc']
            elif bc == None:
              t = t[t['scn'] == scn]
            else:
              t = t[(t['scn'] == scn) & (t['bc'] == 'bc')]

            if pvar in ['TNn', 'TXx', 'avg_tavg', 'avg_tmax', 'avg_tmin']: #temperature related
              color = 'RdBu_r'
            elif pvar in ['CDD', 'CWD', 'FD', 'GSL', 'ID', 'R20mm',
                        'R95p', 'R99p', 'SU', 'csdi', 'days_TN10p', 'days_TX90p', 'hwdi',
                        'max_cold_spell_length', 'wsdi']: #number of days related
              color = 'YlOrRd'
            elif pvar in ['PRCPTOT', 'R20mm_TOT','R95pTOT', 'R99pTOT', 'Rx1day', 'Rx5day',
                          'ann_ppt','RL20', 'RL20_2030_2059', 'RL20_2070_2099','p99wet','p95wet','fwet']:
              color = 'YlGnBu'
            if len(t) > 0:
              mdf = t
            else:
              mdf = None
            print(f'{climate} {group} {period} {pvar} {len(mdf) if mdf is not None else 0}')
            if mdf is not None:
              print((f'{climate} {group} {period} {pvar} {len(t)}'))
              #color = 'PuBuGn'
              zmin,zmax,tick,text = None,None,None,None
              if pvar in ticks_and_color:
                zmin = min(ticks_and_color[pvar]['tick_vals'])
                zmax = max(ticks_and_color[pvar]['tick_vals'])
                tickval = ticks_and_color[pvar]['tick_vals']
                ticktxt = ticks_and_color[pvar]['tick_text']
              fig = mapping_df(mdf
                        ,f'{pvar} ({climate}::{scn}::{period})'
                        ,pvar
                        ,False
                        ,states
                        ,wiras
                        ,ranges[map_region]['lat'][0]
                        ,ranges[map_region]['lat'][1]
                        ,ranges[map_region]['lon'][0]
                        ,ranges[map_region]['lon'][1]
                        ,color
                        ,zmin
                        ,zmax
                        ,tickval
                        ,ticktxt
                        ,True)
              if fig is not None:
                fig.write_image(f'{outroot}/map_{climate}_{scn}_{map_region}_{period}_{pvar}.png', scale=2)

In [ ]:
level_means['grid']['annual']['period'].unique()

In [ ]:
for climate in ['maca_v2','maca3_cmip6','WRF_CMIP6']:
  if climate == 'maca_v2':
    scns = ['rcp45','rcp85']
  else:
    scns = ['ssp370']
  for scn in scns:
    t = level_means['grid']['ref'].copy()
    map_region = 'WA'
    bc = 'bc'
    t = t[(t['climate'] == climate) & (t['scn'] == scn) & (t['bc'] == bc)]
    t['RL20_diff_2040s-1980s'] = t['RL20_2030_2059'] - t['RL20']
    t['RL20_diff_2080s-1980s'] = t['RL20_2070_2099'] - t['RL20']
    t['RL20_diff_2040s-1980s_%'] = (t['RL20_2030_2059'] - t['RL20']) * 100 / t['RL20']
    t['RL20_diff_2080s-1980s_%'] = (t['RL20_2070_2099'] - t['RL20']) * 100 / t['RL20']
    for period in ['2080s-1980s_%', '2080s-1980s','2040s-1980s_%', '2040s-1980s']:
      color = 'BrBG'
      if '%' in period:
        unit = '%'
        tickval = [-5,0,5,10,15,20,25,30,35,40]
      else:
        unit = 'mm/day'
        tickval = [-5,0,5,10,15,20,25,30,35,40]

      data_min, data_max = np.min(tickval), np.max(tickval)
      midpoint = (0 - data_min) / (data_max - data_min)  # = 0.2

      colorscale = [
          [0,          'rgb(180, 0, 0)'],     # dark red (most negative)
          [midpoint,   'rgb(245, 245, 245)'], # white at zero
          [1,          'rgb(0, 0, 180)'],     # dark blue (most positive)
      ]

      zmin = np.min(tickval)
      zmax = np.max(tickval)
      ticktxt = tickval

      fig = mapping_df(t
                ,f'RL20 change ({period[:11]}) ({unit}) {climate} {scn}'
                ,f'RL20_diff_{period}'
                ,False
                ,states
                ,wiras
                ,ranges[map_region]['lat'][0]
                ,ranges[map_region]['lat'][1]
                ,ranges[map_region]['lon'][0]
                ,ranges[map_region]['lon'][1]
                ,colorscale
                ,zmin
                ,zmax
                ,tickval
                ,ticktxt
                ,True)
      fig.write_image(f'{outroot}/map_{climate}_{scn}_{map_region}_{period}_RL20_change.png', scale=2)

In [ ]:
#plot the annual change
var = 'R95pTOT'
for climate in ['maca_v2','WRF_CMIP6']:
  if climate == 'maca_v2':
    scns = ['rcp45','rcp85']
  else:
    scns = ['ssp370']
  for scn in scns:
    t = level_means['grid']['annual'].copy()
    map_region = 'WA'
    bc = 'bc'
    t = t[(t['climate'] == climate) & (t['scn'] == scn) & (t['bc'] == bc)]
    t_1980s = t[t['period'] == '1980s']
    t_2040s = t[t['period'] == '2040s']
    merged = t_2040s.merge(t_1980s, on='gridid', suffixes=('_1', '_2'))
    merged['difference'] = merged[f'{var}_1'] - merged[f'{var}_2']
    merged['1980s'] = merged[f'{var}_2']


    merged['difference_%'] = merged['difference'] * 100 / merged['1980s']
    for period in ['difference', 'difference_%']:
      color = 'BrBG'
      if '%' in period:
        unit = '%'
        tickval = list(range(-20,41,10))
      else:
        unit = 'mm/annual'
        tickval = [-20,-10,0,10,20,40,80,120] #list(range(-20,101,20))

      data_min, data_max = np.min(tickval), np.max(tickval)
      midpoint = (0 - data_min) / (data_max - data_min)  # = 0.2

      colorscale = [
          [0,          'rgb(180, 0, 0)'],     # dark red (most negative)
          [midpoint,   'rgb(245, 245, 245)'], # white at zero
          [1,          'rgb(0, 0, 180)'],     # dark blue (most positive)
      ]

      zmin = np.min(tickval)
      zmax = np.max(tickval)
      ticktxt = tickval

      fig = mapping_df(merged
                ,f'{var} change (2040s-1980s) ({unit}) {climate} {scn}'
                ,f'{period}'
                ,False
                ,states
                ,wiras
                ,ranges[map_region]['lat'][0]
                ,ranges[map_region]['lat'][1]
                ,ranges[map_region]['lon'][0]
                ,ranges[map_region]['lon'][1]
                ,colorscale
                ,zmin
                ,zmax
                ,tickval
                ,ticktxt
                ,True)
      fig.write_image(f'{outroot}/map_{climate}_{scn}_{map_region}_{period}_{var}_change.png', scale=2)

In [ ]:
if False:
  import pandas as pd
  import matplotlib.pyplot as plt
  import matplotlib.cm as cm
  import numpy as np


  MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                  'Jul','Aug','Sep','Oct','Nov','Dec']

  COLORS = [
      '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
      '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
      '#469990','#dcbeff','#9A6324','#fffac8','#800000',
      '#aaffc3','#808000','#ffd8b1','#000075','#a9a9a9'
  ]

  LINE_STYLES = ['-', '--', '-.', ':']
  df = WRIA_final_df['gridmet']['monthly']['1980s']

  # Force sorted order 1–62
  grid_ids = sorted(df['WRIA_NR'].unique())

  # 20 colors × 4 line styles = 80 combinations, enough for 62
  styles = {}
  for i, gid in enumerate(grid_ids):
      color     = COLORS[i % len(COLORS)]
      linestyle = LINE_STYLES[i // len(COLORS)]
      styles[gid] = (color, linestyle)

  fig, ax = plt.subplots(figsize=(14, 6))

  for gid, (color, linestyle) in styles.items():
      subset = df[df['WRIA_NR'] == gid].sort_values('doy_or_month')
      ax.plot(subset['doy_or_month'], subset['mon_ppt'],
              color=color, linestyle=linestyle,
              linewidth=1.4, alpha=0.85, label=str(gid))

  ax.set_xticks(range(1, 13))
  ax.set_xticklabels(MONTH_LABELS, fontsize=11)
  ax.set_xlabel('Month', fontsize=12)
  ax.set_ylabel('Precipitation (mm)', fontsize=12)
  ax.set_title('Monthly Precipitation by WRIA_NR', fontsize=14, fontweight='bold')
  ax.grid(True, linestyle='--', alpha=0.3)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)

  ax.legend(title='WRIA_NR', bbox_to_anchor=(1.01, 1), loc='upper left',
            fontsize=7, title_fontsize=9, framealpha=0.5, ncol=2)

  plt.tight_layout()
  plt.show()

In [ ]:
if False:
  import pandas as pd
  import matplotlib.pyplot as plt
  import numpy as np

  # ── Data ──────────────────────────────────────────────────────────────────────
  # Assumes df is already loaded and sorted by WRIA_NR
  df = WRIA_final_df['gridmet']['ref']['ref']
  df_plot = df.sort_values('WRIA_NR').reset_index(drop=True)

  wria     = df_plot['WRIA_NR'].astype(str)
  x        = np.arange(len(wria))
  variables = ['p99wet', 'p95wet', 'RL20']
  colors    = ['#4363d8', '#e6194b', '#3cb44b']
  titles    = ['p99wet — 99th Percentile Wet', 'p95wet — 95th Percentile Wet', 'RL20 — 20-Year Return Level']

  # ── Plot ──────────────────────────────────────────────────────────────────────
  fig, axes = plt.subplots(3, 1, figsize=(20, 14), sharex=True)
  fig.suptitle('Precipitation Metrics by WRIA_NR', fontsize=16, fontweight='bold', y=1.01)

  for ax, var, color, title in zip(axes, variables, colors, titles):
      bars = ax.bar(x, df_plot[var], color=color, alpha=0.85, width=0.6, edgecolor='white', linewidth=0.4)
      ax.set_ylabel(var, fontsize=11)
      ax.set_title(title, fontsize=12, fontweight='bold', loc='left', pad=6)
      ax.grid(axis='y', linestyle='--', alpha=0.4)
      ax.spines['top'].set_visible(False)
      ax.spines['right'].set_visible(False)
      ax.set_xlim(-0.8, len(x) - 0.2)

      # Value labels on top of each bar
      for bar in bars:
          h = bar.get_height()
          ax.text(bar.get_x() + bar.get_width() / 2, h * 1.01,
                  f'{h:.1f}', ha='center', va='bottom', fontsize=5.5, color='#333333')

  # Shared x-axis labels
  axes[-1].set_xticks(x)
  axes[-1].set_xticklabels(wria, rotation=90, fontsize=8)
  axes[-1].set_xlabel('WRIA_NR', fontsize=12)

  plt.tight_layout()
  plt.show()